# NINA Output CSV for Spectra
## Adjust Celestial Coordinates for Assigned targets


In [1]:
!mkdir -p data_folder

# INITIALIZE CELESTIAL COORDINATES PROCESS

In [2]:
#pip install --upgrade astropy astroquery googlesearch-python wikipedia pandas


In [3]:
from astropy.coordinates import EarthLocation, AltAz, SkyCoord, SkyOffsetFrame
from astropy.time import Time
import astropy.units as u
from astropy.io import ascii
import numpy as np
import pandas as pd
# importing the module
import wikipedia as wiki
from IPython.display import Markdown as md

In [4]:
# ra_dec_offset_v5 and output files naming
celestial_coordinates_version = "v2"
data_folder_name = "data_folder"
output_csvfilename = f"celestial_coordinates_version_{celestial_coordinates_version}.csv"
print(f"celestial_coordinates_version is: {celestial_coordinates_version}")
print(f"data_folder_name is: {data_folder_name}")
print(f"output_csvfilename is: {output_csvfilename}")

celestial_coordinates_version is: v2
data_folder_name is: data_folder
output_csvfilename is: celestial_coordinates_version_v2.csv


In [5]:
from astropy.coordinates import EarthLocation, AltAz, SkyCoord, SkyOffsetFrame
from astropy.time import Time
import astropy.units as u

obs_tel = "BARO"
obs_loc = "San Diego"
obs_lat = 32.6 * u.deg  # for san diego
obs_lon = -116.3 * u.deg # for san diego
obs_hgt = 1131 * u.m # for BARO
safe_lim = 10 * u.deg # Account for BARO Telescop stops 
max_mag = 8 # max star magnitudes to consider
min_ra = 12 # min ra limit for BARO
max_ra = 18 # max ra limit for BARO

print(f"Observers Location is: {obs_loc}")
print(f"Observers Telescope is: {obs_tel}")
print(f"Observers Lattitude is: {obs_lat}")
print(f"Observers Longitude is: {obs_lon}")
print(f"Observers Height is: {obs_hgt}")
print(f"Safe Limit for {obs_tel} is: {safe_lim}")
print(f"Max Mag to Query is: {max_mag}")
print(f"Min RA to Query is: {min_ra}")
print(f"Max RA to Query is: {max_ra}")

# Define observer location
location = EarthLocation.from_geodetic(
    lat=obs_lat, lon=obs_lon, height=obs_hgt
)

# Define observation time
time = Time("2025-06-09 21:30:00")

# Define the celestial object's coordinates (e.g., RA and Dec)
sky_coord = SkyCoord(ra=10 * u.deg, dec=20 * u.deg)

# Create an AltAz frame
altaz_frame = AltAz(obstime=time, location=location)

# Transform the object's coordinates to AltAz
altaz_coord = sky_coord.transform_to(altaz_frame)

# Get the altitude and azimuth
altitude = altaz_coord.alt
azimuth = altaz_coord.az

print(f"Altitude: {altitude:.4f}")
print(f"Azimuth: {azimuth:.4f}")


# Define location and time
#location = EarthLocation(lat='32.7', lon='-116.33', height=0*u.m)
#obstime = Time.now()
#obstime = datetime.time(21, 0)

# AltAz frame for the observer
#altaz_frame = AltAz(obstime=obstime, location=location)

# Determine Declination range
min_dec = location.lat - 90*u.deg + safe_lim
max_dec = location.lat + 90*u.deg - safe_lim
print(f"Observable Declination range: {min_dec.to_string(unit=u.deg)} to {max_dec.to_string(unit=u.deg)}")

# Set global offset for spectra in frame to include zero-order
global_offset_arcmin = 2 # reduced from 3.5
print(f"Global offset RA & Dec arcmin by {global_offset_arcmin} * sin(camera rotation angle)")

Observers Location is: San Diego
Observers Telescope is: BARO
Observers Lattitude is: 32.6 deg
Observers Longitude is: -116.3 deg
Observers Height is: 1131.0 m
Safe Limit for BARO is: 10.0 deg
Max Mag to Query is: 8
Min RA to Query is: 12
Max RA to Query is: 18
Altitude: 7.1947 deg
Azimuth: 289.3402 deg
Observable Declination range: -47d24m00s to 112d36m00s
Global offset RA & Dec arcmin by 2 * sin(camera rotation angle)


In [6]:
obs_zen = 90 * u.deg - obs_lat
safe_min = obs_lat -90 * u.deg + safe_lim
safe_max = obs_lat +90 * u.deg - safe_lim
print(f'The Zenith at {obs_loc} is: {obs_zen:0.2f} deg')
print(f'Safe Declination limits at {obs_tel} are: {safe_min:0.2f} deg to {safe_max:0.2f} deg')

The Zenith at San Diego is: 57.40 deg deg
Safe Declination limits at BARO are: -47.40 deg deg to 112.60 deg deg


In [7]:
# set target default name
target_default_name = "HD"

In [8]:
def compute_exposure_time(mag: float) -> float:
    """
    Compute exposure time (in seconds) to reach 50,000 flux
    given the apparent magnitude, using the refit model
    (excluding La Superba).
    """
    a = 0.9325
    b = 1.0569
    c = -12.325
    target_flux = 50000

    log_flux = np.log(target_flux)
    log_exp = (log_flux + a * mag + c) / b
    return np.exp(log_exp)*2.5



In [9]:
def compute_adj_coord(original_ra_, original_dec_, offset_arcmin_, camera_rotation_deg_): 
    # --- Step 1A: Compute sky position angle for image "left" ---
    original_coord = SkyCoord(original_ra_, original_dec_)
    #original_coord_icrs = original_coord.transform_to('icrs')
    
    # --- Step 2: Compute sky position angle for image "left" ---
    sky_PA = (270 - camera_rotation_deg_) * u.deg
    
    print(f'\nsky_PA: {sky_PA}')
    
    # --- Step 3: Offset distance converted to tangent plane components ---
    offset_dist = offset_arcmin_ * u.arcmin
    
    print(f'\noffset_dist: {offset_dist}')
    
    dx = offset_dist * np.sin(sky_PA)
    dy = offset_dist * np.cos(sky_PA)
    
    print(f'\ndx: {dx} dy: {dy}')
          

    # --- Step 4: Define the offset frame centered on the original target ---
    offset_frame = SkyOffsetFrame(origin=original_coord)
    
    print(f'\noffset_frame = {offset_frame}')

    # --- Step 5: Create a coordinate in the offset frame and transform back ---
    offset_coord = SkyCoord(lon=dx, lat=dy, frame=offset_frame)
    new_coord_ = offset_coord.transform_to('icrs')
    
    print(f'\noffset_coord = {offset_coord}')
    print(f'\nnew_coord_ = {new_coord_}')
    
    return new_coord_



In [10]:
import requests
import time

def make_api_request(url, max_retries=5, initial_delay=2):
    retries = 0
    while retries < max_retries:
        try:
            response = requests.get(url)
            if response.status_code == 429:
                print(f"Received 429 error. Retrying in {initial_delay} seconds...")
                retry_after = response.headers.get('Retry-After')
                if retry_after:
                    delay = int(retry_after)
                else:
                    delay = initial_delay * (2 ** retries) # Exponential backoff
                time.sleep(delay)
                retries += 1
            elif response.status_code == 200:
                return response.json()
            else:
                print(f"Error: {response.status_code}")
                return None
        except requests.exceptions.RequestException as e:
            print(f"Request failed: {e}")
            time.sleep(initial_delay * (2 ** retries)) # Exponential backoff for network errors
            retries += 1
    print("Max retries exceeded. Request failed.")
    return None

# wikipedia Example usage
# to search
#query = f'https://en.wikipedia.org/wiki/{target_name} Wikipedia Astronomy'

#results = make_api_request(query)
#if results:
#    print("Data received:", results)


In [11]:
def googlesearchurl(tname):
    try:
        from googlesearch import search
    except ImportError:
        print("No module named 'google' found")
    #
    # to search
    query = f'{tname} Wikipedia Astronomy'
    
    for j in search(query, tld="co.us", num=10, stop=10, pause=2):
            print(j)
    #for url in search(query, num_results=10, lang="en", unique=True):
    #    print(url)
#
# example
#
#tname = "RR Lyrae"
#googlesearchurl(tname)

In [12]:
def wikisearchurl(tname):
    # Set language to English (optional)
    wiki.set_lang("en")
    
    # Search for a topic
    results = wiki.search(tname)
    print("Search Results:", results)
    
    # Get a summary of the first result
    summary = wiki.summary(results[0], sentences=1)
    print("\nSummary:", summary)
    
    # Get the full page object
    page = wiki.page(results[0])
    print("\nPage Title:", page.title)
    print("Page URL:", page.url)
#
# example
#
#tname = "RR Lyrae"
#wikisearchurl(tname)

In [13]:
def obtain_info_for_adhoc_star(tname, ara, adec):
    # --- Change User Inputs ---
    #offset_arcmin = -3.5                    # Offset distance (arcmin)
    offset_arcmin = global_offset_arcmin
    camera_rotation_deg = -21.9              # 1/12 of a full rotation = 30° clockwise
    
    # --- Step 1: Look up target coordinates ---
    #original_coord = SkyCoord.from_name(tname)
    original_coord = SkyCoord(ara, adec, frame='icrs', unit='deg')
    
    print(original_coord)
    
    # --- Step 1A-5 : Call function to calculate adjusted coordinates ---
    new_coord = compute_adj_coord(original_coord.ra, original_coord.dec, offset_arcmin, camera_rotation_deg)
    
    # --- Step 6: Report result ---
    print(f"Using SkyOffsetFrame for Star {tname} ")
    print(f"User inputs: offset_arcmin = {offset_arcmin}, camera_rotation_deg = {camera_rotation_deg}")
    print(f"\nOriginal RA(Hr)/Dec: {original_coord.ra.hour:.4f}, {original_coord.dec.deg:.4f}")
    print(f"Original RA(Deg)/Dec: {original_coord.ra.deg:.4f}, {original_coord.dec.deg:.4f}")
    print(f"New  RA(Hr)/Dec:  {new_coord.ra.hour:.4f}, {new_coord.dec.deg:.4f}")
    print(f"New  RA(Deg)/Dec:  {new_coord.ra.deg:.4f}, {new_coord.dec.deg:.4f}")
    print(f"Delta RA/DEC(min): {(original_coord.ra.deg-new_coord.ra.deg):.4f}, \
          {(original_coord.dec.deg-new_coord.dec.deg):.4f}")
    
    #print("\n");wikisearchurl(tname);print("\n")

    return(new_coord)

In [14]:
def obtain_info_for_star(tname):
    # --- Change User Inputs ---
    #offset_arcmin = -3.5                    # Offset distance (arcmin)
    offset_arcmin = global_offset_arcmin
    camera_rotation_deg = -21.9              # 1/12 of a full rotation = 30° clockwise
    
    # --- Step 1: Look up target coordinates ---
    original_coord = SkyCoord.from_name(tname)
    
    print(original_coord)
    
    # --- Step 1A-5 : Call function to calculate adjusted coordinates ---
    new_coord = compute_adj_coord(original_coord.ra, original_coord.dec, offset_arcmin, camera_rotation_deg)
    
    # --- Step 6: Report result ---
    print(f"Using SkyOffsetFrame for Star {tname} ")
    print(f"User inputs: offset_arcmin = {offset_arcmin}, camera_rotation_deg = {camera_rotation_deg}")
    print(f"\nOriginal RA(Hr)/Dec: {original_coord.ra.hour:.4f}, {original_coord.dec.deg:.4f}")
    print(f"Original RA(Deg)/Dec: {original_coord.ra.deg:.4f}, {original_coord.dec.deg:.4f}")
    print(f"New  RA(Hr)/Dec:  {new_coord.ra.hour:.4f}, {new_coord.dec.deg:.4f}")
    print(f"New  RA(Deg)/Dec:  {new_coord.ra.deg:.4f}, {new_coord.dec.deg:.4f}")
    print(f"Delta RA/DEC(min): {(original_coord.ra.deg-new_coord.ra.deg):.4f}, \
          {(original_coord.dec.deg-new_coord.dec.deg):.4f}")
    
    #print("\n");wikisearchurl(tname);print("\n")

    return(new_coord, original_coord)

In [15]:
def obtain_adjcoord_for_planet(pname, pcoord):
    # --- Change User Inputs ---
    #offset_arcmin = -3.5                    # Offset distance (arcmin)
    offset_arcmin = global_offset_arcmin
    camera_rotation_deg = -21.9              # 1/12 of a full rotation = 30° clockwise
    
    # --- Step 1: Look up target coordinates ---
    #original_coord = SkyCoord.from_name(target_name)
    
    print(pcoord)
    
    # --- Step 1A-5 : Call function to calculate adjusted coordinates ---
    new_coord = compute_adj_coord(pcoord.ra, pcoord.dec, offset_arcmin, camera_rotation_deg)
    
    # --- Step 6: Report result ---
    print(f"Using SkyOffsetFrame for Star {target_name} ")
    print(f"User inputs: offset_arcmin = {offset_arcmin}, camera_rotation_deg = {camera_rotation_deg}")
    print(f"\nOriginal RA(Hr)/Dec: {pcoord.ra.hour:.4f}, {pcoord.dec.deg:.4f}")
    print(f"Original RA(Deg)/Dec: {pcoord.ra.deg:.4f}, {pcoord.dec.deg:.4f}")
    print(f"New  RA(Hr)/Dec:  {new_coord.ra.hour:.4f}, {new_coord.dec.deg:.4f}")
    print(f"New  RA(Deg)/Dec:  {new_coord.ra.deg:.4f}, {new_coord.dec.deg:.4f}")
    print(f"Delta RA/DEC(min): {(pcoord.ra.deg-new_coord.ra.deg):.4f}, \
          {(pcoord.dec.deg-new_coord.dec.deg):.4f}")
    
    #print("\n");googlesearchurl(target_name);print("\n")

    return(new_coord, orig_coord)

In [16]:
# ---  CREATE A DATAFRAME
column_names = ["Name1*","Name2*","RA2000*","D2000*","Pmag~","Exp~","Note1","Note2","NExp~","GetRef","Temp","rahrdec","rahrdegdec","decdegdec"] 
df = pd.DataFrame(columns=column_names)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)  # Or a large integer like 9999
ridx = 0

# CELESTIAL COORDINATES FOR NEW STAR 3C273

In [17]:
# CELESTIAL COORDINATES FOR NEW STAR

from IPython.display import Markdown as md
# Instead of setting the cell to Markdown, create Markdown from withnin a code cell!
# We can just use python variable replacement syntax to make the text dynamic
target_name = "3C273"
md(f"### Obtain spectral details for new star {target_name}")


### Obtain spectral details for new star 3C273

In [18]:
new_coord, orig_coord = obtain_info_for_star(target_name)

<SkyCoord (ICRS): (ra, dec) in deg
    (187.27791594, 2.05238823)>

sky_PA: 291.9 deg

offset_dist: 2.0 arcmin

dx: -1.8556725077978407 arcmin dy: 0.7459755651516162 arcmin

offset_frame = <SkyOffsetICRS Frame (rotation=0.0 deg, origin=<ICRS Coordinate: (ra, dec) in deg
    (187.27791594, 2.05238823)>)>

offset_coord = <SkyCoord (SkyOffsetICRS: rotation=0.0 deg, origin=<ICRS Coordinate: (ra, dec) in deg
    (187.27791594, 2.05238823)>): (lon, lat) in deg
    (-0.03092788, 0.01243293)>

new_coord_ = <SkyCoord (ICRS): (ra, dec) in deg
    (187.24696797, 2.06482086)>
Using SkyOffsetFrame for Star 3C273 
User inputs: offset_arcmin = 2, camera_rotation_deg = -21.9

Original RA(Hr)/Dec: 12.4852, 2.0524
Original RA(Deg)/Dec: 187.2779, 2.0524
New  RA(Hr)/Dec:  12.4831, 2.0648
New  RA(Deg)/Dec:  187.2470, 2.0648
Delta RA/DEC(min): 0.0309,           -0.0124


### Update Properties for Star based on wiki query results above 

In [19]:
star_magnitude = 12.9;  target_alt_name = "Quasar_3C273" # FIX THIS LINE AND BELOW BASED ON SEARCH
star_type = 'Qu'
exposure = compute_exposure_time(star_magnitude)
print(f"Required exposure for mag {star_magnitude}: {exposure:.2f} s")

Required exposure for mag 12.9: 52772.22 s


In [20]:
df.loc[ridx]=[f'{target_name}_{target_alt_name}_Typ_{star_type}',f'{target_alt_name}',f'{new_coord.ra.deg:.4f}',f'{new_coord.dec.deg:.4f}',
           f'{star_magnitude}',f'{exposure:.2f}','NA','NA','1','0','',f'{orig_coord.ra.hour:.4f}',f'{orig_coord.ra.deg:.4f}',f'{orig_coord.dec.deg:.4f}']
ridx += 1
df['Name1*'] = df['Name1*'].str.replace(' ', '_'); print(df[['Name1*','Name2*',"RA2000*","D2000*","Pmag~","Exp~","rahrdec","rahrdegdec","decdegdec"]])

                      Name1*        Name2*   RA2000*  D2000* Pmag~      Exp~  \
0  3C273_Quasar_3C273_Typ_Qu  Quasar_3C273  187.2470  2.0648  12.9  52772.22   

   rahrdec rahrdegdec decdegdec  
0  12.4852   187.2779    2.0524  


# OUTPUT CONSOLIDATED CSV FILE FOR BARO

In [21]:
df.to_csv(f"{data_folder_name}/{output_csvfilename}")
print(df[["Name1*","Name2*","RA2000*","D2000*","Pmag~","Exp~","Note1","Note2","NExp~","GetRef","Temp","rahrdec","rahrdegdec","decdegdec"]])

                      Name1*        Name2*   RA2000*  D2000* Pmag~      Exp~  \
0  3C273_Quasar_3C273_Typ_Qu  Quasar_3C273  187.2470  2.0648  12.9  52772.22   

  Note1 Note2 NExp~ GetRef Temp  rahrdec rahrdegdec decdegdec  
0    NA    NA     1      0       12.4852   187.2779    2.0524  


<div class="alert alert-danger"><strong>STOP HERE AS NEEDED FOR CONCISE CSV TARGETS</strong></div>

# CELESTIAL COORDINATES FOR NEW STAR HD108959

In [22]:
# CELESTIAL COORDINATES FOR NEW STAR

from IPython.display import Markdown as md
# Instead of setting the cell to Markdown, create Markdown from withnin a code cell!
# We can just use python variable replacement syntax to make the text dynamic
target_name = "HD 108959"
md(f"### Obtain spectral details for new star {target_name}")


### Obtain spectral details for new star HD 108959

In [23]:
new_coord, orig_coord = obtain_info_for_star(target_name)

<SkyCoord (ICRS): (ra, dec) in deg
    (187.8108921, 1.32695953)>

sky_PA: 291.9 deg

offset_dist: 2.0 arcmin

dx: -1.8556725077978407 arcmin dy: 0.7459755651516162 arcmin

offset_frame = <SkyOffsetICRS Frame (rotation=0.0 deg, origin=<ICRS Coordinate: (ra, dec) in deg
    (187.8108921, 1.32695953)>)>

offset_coord = <SkyCoord (SkyOffsetICRS: rotation=0.0 deg, origin=<ICRS Coordinate: (ra, dec) in deg
    (187.8108921, 1.32695953)>): (lon, lat) in deg
    (-0.03092788, 0.01243293)>

new_coord_ = <SkyCoord (ICRS): (ra, dec) in deg
    (187.77995577, 1.33939226)>
Using SkyOffsetFrame for Star HD 108959 
User inputs: offset_arcmin = 2, camera_rotation_deg = -21.9

Original RA(Hr)/Dec: 12.5207, 1.3270
Original RA(Deg)/Dec: 187.8109, 1.3270
New  RA(Hr)/Dec:  12.5187, 1.3394
New  RA(Deg)/Dec:  187.7800, 1.3394
Delta RA/DEC(min): 0.0309,           -0.0124


### Update Properties for Star based on wiki query results above 

In [24]:
star_magnitude = 8;  target_alt_name = "HD108959" # FIX THIS LINE AND BELOW BASED ON SEARCH
star_type = 'F0IV'
exposure = compute_exposure_time(star_magnitude)
print(f"Required exposure for mag {star_magnitude}: {exposure:.2f} s")

Required exposure for mag 8: 699.58 s


In [25]:
df.loc[ridx]=[f'{target_name}_{target_alt_name}_Typ_{star_type}',f'{target_alt_name}',f'{new_coord.ra.deg:.4f}',f'{new_coord.dec.deg:.4f}',
           f'{star_magnitude}',f'{exposure:.2f}','NA','NA','1','0','',f'{orig_coord.ra.hour:.4f}',f'{orig_coord.ra.deg:.4f}',f'{orig_coord.dec.deg:.4f}']
ridx += 1
df['Name1*'] = df['Name1*'].str.replace(' ', '_'); print(df[['Name1*','Name2*',"RA2000*","D2000*","Pmag~","Exp~"]])

                        Name1*        Name2*   RA2000*  D2000* Pmag~      Exp~
0    3C273_Quasar_3C273_Typ_Qu  Quasar_3C273  187.2470  2.0648  12.9  52772.22
1  HD_108959_HD108959_Typ_F0IV      HD108959  187.7800  1.3394     8    699.58


# CELESTIAL COORDINATES FOR NEW STAR RASALHAGUE

In [26]:
# CELESTIAL COORDINATES FOR NEW STAR

from IPython.display import Markdown as md
# Instead of setting the cell to Markdown, create Markdown from withnin a code cell!
# We can just use python variable replacement syntax to make the text dynamic
target_name = "Rasalhague"
md(f"### Obtain spectral details for new star {target_name}")


### Obtain spectral details for new star Rasalhague

In [27]:
new_coord, orig_coord = obtain_info_for_star(target_name)

<SkyCoord (ICRS): (ra, dec) in deg
    (263.73362272, 12.56003739)>

sky_PA: 291.9 deg

offset_dist: 2.0 arcmin

dx: -1.8556725077978407 arcmin dy: 0.7459755651516162 arcmin

offset_frame = <SkyOffsetICRS Frame (rotation=0.0 deg, origin=<ICRS Coordinate: (ra, dec) in deg
    (263.73362272, 12.56003739)>)>

offset_coord = <SkyCoord (SkyOffsetICRS: rotation=0.0 deg, origin=<ICRS Coordinate: (ra, dec) in deg
    (263.73362272, 12.56003739)>): (lon, lat) in deg
    (-0.03092788, 0.01243293)>

new_coord_ = <SkyCoord (ICRS): (ra, dec) in deg
    (263.70193502, 12.57246846)>
Using SkyOffsetFrame for Star Rasalhague 
User inputs: offset_arcmin = 2, camera_rotation_deg = -21.9

Original RA(Hr)/Dec: 17.5822, 12.5600
Original RA(Deg)/Dec: 263.7336, 12.5600
New  RA(Hr)/Dec:  17.5801, 12.5725
New  RA(Deg)/Dec:  263.7019, 12.5725
Delta RA/DEC(min): 0.0317,           -0.0124


### Update Properties for Star based on wiki query results above 

In [28]:
star_magnitude = 2.07;  target_alt_name = "HD 159561" # FIX THIS LINE AND BELOW BASED ON SEARCH
star_type = 'A5'
exposure = compute_exposure_time(star_magnitude)
print(f"Required exposure for mag {star_magnitude}: {exposure:.2f} s")

Required exposure for mag 2.07: 3.74 s


In [29]:
df.loc[ridx]=[f'{target_name}_{target_alt_name}_Typ_{star_type}',f'{target_alt_name}',f'{new_coord.ra.deg:.4f}',f'{new_coord.dec.deg:.4f}',
           f'{star_magnitude}',f'{exposure:.2f}','NA','NA','1','0','',f'{orig_coord.ra.hour:.4f}',f'{orig_coord.ra.deg:.4f}',f'{orig_coord.dec.deg:.4f}']
ridx += 1
df['Name1*'] = df['Name1*'].str.replace(' ', '_'); print(df[['Name1*','Name2*',"RA2000*","D2000*","Pmag~","Exp~"]])

                        Name1*        Name2*   RA2000*   D2000* Pmag~  \
0    3C273_Quasar_3C273_Typ_Qu  Quasar_3C273  187.2470   2.0648  12.9   
1  HD_108959_HD108959_Typ_F0IV      HD108959  187.7800   1.3394     8   
2  Rasalhague_HD_159561_Typ_A5     HD 159561  263.7019  12.5725  2.07   

       Exp~  
0  52772.22  
1    699.58  
2      3.74  


# CELESTIAL COORDINATES FOR NEW STAR RASALGETHI

In [30]:
# CELESTIAL COORDINATES FOR NEW STAR

from IPython.display import Markdown as md
# Instead of setting the cell to Markdown, create Markdown from withnin a code cell!
# We can just use python variable replacement syntax to make the text dynamic
target_name = "Rasalgethi"
md(f"### Obtain spectral details for new star {target_name}")


### Obtain spectral details for new star Rasalgethi

In [31]:
new_coord, orig_coord = obtain_info_for_star(target_name)

<SkyCoord (ICRS): (ra, dec) in deg
    (258.66190909, 14.39034061)>

sky_PA: 291.9 deg

offset_dist: 2.0 arcmin

dx: -1.8556725077978407 arcmin dy: 0.7459755651516162 arcmin

offset_frame = <SkyOffsetICRS Frame (rotation=0.0 deg, origin=<ICRS Coordinate: (ra, dec) in deg
    (258.66190909, 14.39034061)>)>

offset_coord = <SkyCoord (SkyOffsetICRS: rotation=0.0 deg, origin=<ICRS Coordinate: (ra, dec) in deg
    (258.66190909, 14.39034061)>): (lon, lat) in deg
    (-0.03092788, 0.01243293)>

new_coord_ = <SkyCoord (ICRS): (ra, dec) in deg
    (258.62997765, 14.40277139)>
Using SkyOffsetFrame for Star Rasalgethi 
User inputs: offset_arcmin = 2, camera_rotation_deg = -21.9

Original RA(Hr)/Dec: 17.2441, 14.3903
Original RA(Deg)/Dec: 258.6619, 14.3903
New  RA(Hr)/Dec:  17.2420, 14.4028
New  RA(Deg)/Dec:  258.6300, 14.4028
Delta RA/DEC(min): 0.0319,           -0.0124


### Update Properties for Star based on wiki query results above 

In [32]:
star_magnitude = 5.3;  target_alt_name = "HD 156014" # FIX THIS LINE AND BELOW BASED ON SEARCH
star_type = 'M5'
exposure = compute_exposure_time(star_magnitude)
print(f"Required exposure for mag {star_magnitude}: {exposure:.2f} s")

Required exposure for mag 5.3: 64.60 s


In [33]:
df.loc[ridx]=[f'{target_name}_{target_alt_name}_Typ_{star_type}',f'{target_alt_name}',f'{new_coord.ra.deg:.4f}',f'{new_coord.dec.deg:.4f}',
           f'{star_magnitude}',f'{exposure:.2f}','NA','NA','1','0','',f'{orig_coord.ra.hour:.4f}',f'{orig_coord.ra.deg:.4f}',f'{orig_coord.dec.deg:.4f}']
ridx += 1
df['Name1*'] = df['Name1*'].str.replace(' ', '_'); print(df[['Name1*','Name2*',"RA2000*","D2000*","Pmag~","Exp~"]])

                        Name1*        Name2*   RA2000*   D2000* Pmag~  \
0    3C273_Quasar_3C273_Typ_Qu  Quasar_3C273  187.2470   2.0648  12.9   
1  HD_108959_HD108959_Typ_F0IV      HD108959  187.7800   1.3394     8   
2  Rasalhague_HD_159561_Typ_A5     HD 159561  263.7019  12.5725  2.07   
3  Rasalgethi_HD_156014_Typ_M5     HD 156014  258.6300  14.4028   5.3   

       Exp~  
0  52772.22  
1    699.58  
2      3.74  
3     64.60  


# CELESTIAL COORDINATES FOR NEW STAR ALTAIR

In [34]:
# CELESTIAL COORDINATES FOR NEW STAR

from IPython.display import Markdown as md
# Instead of setting the cell to Markdown, create Markdown from withnin a code cell!
# We can just use python variable replacement syntax to make the text dynamic
target_name = "Altair"
md(f"### Obtain spectral details for new star {target_name}")


### Obtain spectral details for new star Altair

In [35]:
new_coord, orig_coord = obtain_info_for_star(target_name)

<SkyCoord (ICRS): (ra, dec) in deg
    (297.6958273, 8.8683212)>

sky_PA: 291.9 deg

offset_dist: 2.0 arcmin

dx: -1.8556725077978407 arcmin dy: 0.7459755651516162 arcmin

offset_frame = <SkyOffsetICRS Frame (rotation=0.0 deg, origin=<ICRS Coordinate: (ra, dec) in deg
    (297.6958273, 8.8683212)>)>

offset_coord = <SkyCoord (SkyOffsetICRS: rotation=0.0 deg, origin=<ICRS Coordinate: (ra, dec) in deg
    (297.6958273, 8.8683212)>): (lon, lat) in deg
    (-0.03092788, 0.01243293)>

new_coord_ = <SkyCoord (ICRS): (ra, dec) in deg
    (297.66452415, 8.88075282)>
Using SkyOffsetFrame for Star Altair 
User inputs: offset_arcmin = 2, camera_rotation_deg = -21.9

Original RA(Hr)/Dec: 19.8464, 8.8683
Original RA(Deg)/Dec: 297.6958, 8.8683
New  RA(Hr)/Dec:  19.8443, 8.8808
New  RA(Deg)/Dec:  297.6645, 8.8808
Delta RA/DEC(min): 0.0313,           -0.0124


### Update Properties for Star based on wiki query results above 

In [36]:
star_magnitude = 0.76;  target_alt_name = "HD 187642" # FIX THIS LINE AND BELOW BASED ON SEARCH
star_type = 'A7'
exposure = compute_exposure_time(star_magnitude)
print(f"Required exposure for mag {star_magnitude}: {exposure:.2f} s")

Required exposure for mag 0.76: 1.18 s


In [37]:
df.loc[ridx]=[f'{target_name}_{target_alt_name}_Typ_{star_type}',f'{target_alt_name}',f'{new_coord.ra.deg:.4f}',f'{new_coord.dec.deg:.4f}',
           f'{star_magnitude}',f'{exposure:.2f}','NA','NA','1','0','',f'{orig_coord.ra.hour:.4f}',f'{orig_coord.ra.deg:.4f}',f'{orig_coord.dec.deg:.4f}']
ridx += 1
df['Name1*'] = df['Name1*'].str.replace(' ', '_'); print(df[['Name1*','Name2*',"RA2000*","D2000*","Pmag~","Exp~"]])

                        Name1*        Name2*   RA2000*   D2000* Pmag~  \
0    3C273_Quasar_3C273_Typ_Qu  Quasar_3C273  187.2470   2.0648  12.9   
1  HD_108959_HD108959_Typ_F0IV      HD108959  187.7800   1.3394     8   
2  Rasalhague_HD_159561_Typ_A5     HD 159561  263.7019  12.5725  2.07   
3  Rasalgethi_HD_156014_Typ_M5     HD 156014  258.6300  14.4028   5.3   
4      Altair_HD_187642_Typ_A7     HD 187642  297.6645   8.8808  0.76   

       Exp~  
0  52772.22  
1    699.58  
2      3.74  
3     64.60  
4      1.18  


# CELESTIAL COORDINATES FOR NEW STAR Vega

In [38]:
# CELESTIAL COORDINATES FOR NEW STAR

from IPython.display import Markdown as md
# Instead of setting the cell to Markdown, create Markdown from withnin a code cell!
# We can just use python variable replacement syntax to make the text dynamic
target_name = "Vega"
md(f"### Obtain spectral details for new star {target_name}")


### Obtain spectral details for new star Vega

In [39]:
new_coord, orig_coord = obtain_info_for_star(target_name)

<SkyCoord (ICRS): (ra, dec) in deg
    (279.23473479, 38.78368896)>

sky_PA: 291.9 deg

offset_dist: 2.0 arcmin

dx: -1.8556725077978407 arcmin dy: 0.7459755651516162 arcmin

offset_frame = <SkyOffsetICRS Frame (rotation=0.0 deg, origin=<ICRS Coordinate: (ra, dec) in deg
    (279.23473479, 38.78368896)>)>

offset_coord = <SkyCoord (SkyOffsetICRS: rotation=0.0 deg, origin=<ICRS Coordinate: (ra, dec) in deg
    (279.23473479, 38.78368896)>): (lon, lat) in deg
    (-0.03092788, 0.01243293)>

new_coord_ = <SkyCoord (ICRS): (ra, dec) in deg
    (279.19505214, 38.79611517)>
Using SkyOffsetFrame for Star Vega 
User inputs: offset_arcmin = 2, camera_rotation_deg = -21.9

Original RA(Hr)/Dec: 18.6156, 38.7837
Original RA(Deg)/Dec: 279.2347, 38.7837
New  RA(Hr)/Dec:  18.6130, 38.7961
New  RA(Deg)/Dec:  279.1951, 38.7961
Delta RA/DEC(min): 0.0397,           -0.0124


### Update Properties for Star based on wiki query results above 

In [40]:
star_magnitude = 0.026;  target_alt_name = "HD 172167" # FIX THIS LINE AND BELOW BASED ON SEARCH
star_type = 'A0'
exposure = compute_exposure_time(star_magnitude)
print(f"Required exposure for mag {star_magnitude}: {exposure:.2f} s")

Required exposure for mag 0.026: 0.62 s


In [41]:
df.loc[ridx]=[f'{target_name}_{target_alt_name}_Typ_{star_type}',f'{target_alt_name}',f'{new_coord.ra.deg:.4f}',f'{new_coord.dec.deg:.4f}',
           f'{star_magnitude}',f'{exposure:.2f}','NA','NA','1','0','',f'{orig_coord.ra.hour:.4f}',f'{orig_coord.ra.deg:.4f}',f'{orig_coord.dec.deg:.4f}']
ridx += 1
df['Name1*'] = df['Name1*'].str.replace(' ', '_'); print(df[['Name1*','Name2*',"RA2000*","D2000*","Pmag~","Exp~"]])

                        Name1*        Name2*   RA2000*   D2000*  Pmag~  \
0    3C273_Quasar_3C273_Typ_Qu  Quasar_3C273  187.2470   2.0648   12.9   
1  HD_108959_HD108959_Typ_F0IV      HD108959  187.7800   1.3394      8   
2  Rasalhague_HD_159561_Typ_A5     HD 159561  263.7019  12.5725   2.07   
3  Rasalgethi_HD_156014_Typ_M5     HD 156014  258.6300  14.4028    5.3   
4      Altair_HD_187642_Typ_A7     HD 187642  297.6645   8.8808   0.76   
5        Vega_HD_172167_Typ_A0     HD 172167  279.1951  38.7961  0.026   

       Exp~  
0  52772.22  
1    699.58  
2      3.74  
3     64.60  
4      1.18  
5      0.62  


# CELESTIAL COORDINATES FOR NEW STAR Dubhe

In [42]:
# CELESTIAL COORDINATES FOR NEW STAR

from IPython.display import Markdown as md
# Instead of setting the cell to Markdown, create Markdown from withnin a code cell!
# We can just use python variable replacement syntax to make the text dynamic
target_name = "Dubhe"
md(f"### Obtain spectral details for new star {target_name}")


### Obtain spectral details for new star Dubhe

In [43]:
new_coord, orig_coord = obtain_info_for_star(target_name)

<SkyCoord (ICRS): (ra, dec) in deg
    (165.93196467, 61.75103469)>

sky_PA: 291.9 deg

offset_dist: 2.0 arcmin

dx: -1.8556725077978407 arcmin dy: 0.7459755651516162 arcmin

offset_frame = <SkyOffsetICRS Frame (rotation=0.0 deg, origin=<ICRS Coordinate: (ra, dec) in deg
    (165.93196467, 61.75103469)>)>

offset_coord = <SkyCoord (SkyOffsetICRS: rotation=0.0 deg, origin=<ICRS Coordinate: (ra, dec) in deg
    (165.93196467, 61.75103469)>): (lon, lat) in deg
    (-0.03092788, 0.01243293)>

new_coord_ = <SkyCoord (ICRS): (ra, dec) in deg
    (165.86659363, 61.76345207)>
Using SkyOffsetFrame for Star Dubhe 
User inputs: offset_arcmin = 2, camera_rotation_deg = -21.9

Original RA(Hr)/Dec: 11.0621, 61.7510
Original RA(Deg)/Dec: 165.9320, 61.7510
New  RA(Hr)/Dec:  11.0578, 61.7635
New  RA(Deg)/Dec:  165.8666, 61.7635
Delta RA/DEC(min): 0.0654,           -0.0124


### Update Properties for Star based on wiki query results above 

In [44]:
star_magnitude = 1.79;  target_alt_name = "HD 95689" # FIX THIS LINE AND BELOW BASED ON SEARCH
star_type = 'K0'
exposure = compute_exposure_time(star_magnitude)
print(f"Required exposure for mag {star_magnitude}: {exposure:.2f} s")

Required exposure for mag 1.79: 2.92 s


In [45]:
df.loc[ridx]=[f'{target_name}_{target_alt_name}_Typ_{star_type}',f'{target_alt_name}',f'{new_coord.ra.deg:.4f}',f'{new_coord.dec.deg:.4f}',
           f'{star_magnitude}',f'{exposure:.2f}','NA','NA','1','0','',f'{orig_coord.ra.hour:.4f}',f'{orig_coord.ra.deg:.4f}',f'{orig_coord.dec.deg:.4f}']
ridx += 1
df['Name1*'] = df['Name1*'].str.replace(' ', '_'); print(df[['Name1*','Name2*',"RA2000*","D2000*","Pmag~","Exp~"]])

                        Name1*        Name2*   RA2000*   D2000*  Pmag~  \
0    3C273_Quasar_3C273_Typ_Qu  Quasar_3C273  187.2470   2.0648   12.9   
1  HD_108959_HD108959_Typ_F0IV      HD108959  187.7800   1.3394      8   
2  Rasalhague_HD_159561_Typ_A5     HD 159561  263.7019  12.5725   2.07   
3  Rasalgethi_HD_156014_Typ_M5     HD 156014  258.6300  14.4028    5.3   
4      Altair_HD_187642_Typ_A7     HD 187642  297.6645   8.8808   0.76   
5        Vega_HD_172167_Typ_A0     HD 172167  279.1951  38.7961  0.026   
6        Dubhe_HD_95689_Typ_K0      HD 95689  165.8666  61.7635   1.79   

       Exp~  
0  52772.22  
1    699.58  
2      3.74  
3     64.60  
4      1.18  
5      0.62  
6      2.92  


# CELESTIAL COORDINATES FOR NEW STAR Scheat

In [46]:
# CELESTIAL COORDINATES FOR NEW STAR

from IPython.display import Markdown as md
# Instead of setting the cell to Markdown, create Markdown from withnin a code cell!
# We can just use python variable replacement syntax to make the text dynamic
target_name = "Scheat"
md(f"### Obtain spectral details for new star {target_name}")


### Obtain spectral details for new star Scheat

In [47]:
new_coord, orig_coord = obtain_info_for_star(target_name)

<SkyCoord (ICRS): (ra, dec) in deg
    (345.94357274, 28.08278712)>

sky_PA: 291.9 deg

offset_dist: 2.0 arcmin

dx: -1.8556725077978407 arcmin dy: 0.7459755651516162 arcmin

offset_frame = <SkyOffsetICRS Frame (rotation=0.0 deg, origin=<ICRS Coordinate: (ra, dec) in deg
    (345.94357274, 28.08278712)>)>

offset_coord = <SkyCoord (SkyOffsetICRS: rotation=0.0 deg, origin=<ICRS Coordinate: (ra, dec) in deg
    (345.94357274, 28.08278712)>): (lon, lat) in deg
    (-0.03092788, 0.01243293)>

new_coord_ = <SkyCoord (ICRS): (ra, dec) in deg
    (345.90851373, 28.0952156)>
Using SkyOffsetFrame for Star Scheat 
User inputs: offset_arcmin = 2, camera_rotation_deg = -21.9

Original RA(Hr)/Dec: 23.0629, 28.0828
Original RA(Deg)/Dec: 345.9436, 28.0828
New  RA(Hr)/Dec:  23.0606, 28.0952
New  RA(Deg)/Dec:  345.9085, 28.0952
Delta RA/DEC(min): 0.0351,           -0.0124


### Update Properties for Star based on wiki query results above 

In [48]:
star_magnitude = 2.42;  target_alt_name = "HD 217906" # FIX THIS LINE AND BELOW BASED ON SEARCH
star_type = 'M2'
exposure = compute_exposure_time(star_magnitude)
print(f"Required exposure for mag {star_magnitude}: {exposure:.2f} s")

Required exposure for mag 2.42: 5.09 s


In [49]:
df.loc[ridx]=[f'{target_name}_{target_alt_name}_Typ_{star_type}',f'{target_alt_name}',f'{new_coord.ra.deg:.4f}',f'{new_coord.dec.deg:.4f}',
           f'{star_magnitude}',f'{exposure:.2f}','NA','NA','1','0','',f'{orig_coord.ra.hour:.4f}',f'{orig_coord.ra.deg:.4f}',f'{orig_coord.dec.deg:.4f}']
ridx += 1
df['Name1*'] = df['Name1*'].str.replace(' ', '_'); print(df[['Name1*','Name2*',"RA2000*","D2000*","Pmag~","Exp~"]])

                        Name1*        Name2*   RA2000*   D2000*  Pmag~  \
0    3C273_Quasar_3C273_Typ_Qu  Quasar_3C273  187.2470   2.0648   12.9   
1  HD_108959_HD108959_Typ_F0IV      HD108959  187.7800   1.3394      8   
2  Rasalhague_HD_159561_Typ_A5     HD 159561  263.7019  12.5725   2.07   
3  Rasalgethi_HD_156014_Typ_M5     HD 156014  258.6300  14.4028    5.3   
4      Altair_HD_187642_Typ_A7     HD 187642  297.6645   8.8808   0.76   
5        Vega_HD_172167_Typ_A0     HD 172167  279.1951  38.7961  0.026   
6        Dubhe_HD_95689_Typ_K0      HD 95689  165.8666  61.7635   1.79   
7      Scheat_HD_217906_Typ_M2     HD 217906  345.9085  28.0952   2.42   

       Exp~  
0  52772.22  
1    699.58  
2      3.74  
3     64.60  
4      1.18  
5      0.62  
6      2.92  
7      5.09  


# CELESTIAL COORDINATES FOR NEW STAR Mizar

In [50]:
# CELESTIAL COORDINATES FOR NEW STAR

from IPython.display import Markdown as md
# Instead of setting the cell to Markdown, create Markdown from withnin a code cell!
# We can just use python variable replacement syntax to make the text dynamic
target_name = "Mizar"
md(f"### Obtain spectral details for new star {target_name}")


### Obtain spectral details for new star Mizar

In [51]:
new_coord, orig_coord = obtain_info_for_star(target_name)

<SkyCoord (ICRS): (ra, dec) in deg
    (200.98141867, 54.92535197)>

sky_PA: 291.9 deg

offset_dist: 2.0 arcmin

dx: -1.8556725077978407 arcmin dy: 0.7459755651516162 arcmin

offset_frame = <SkyOffsetICRS Frame (rotation=0.0 deg, origin=<ICRS Coordinate: (ra, dec) in deg
    (200.98141867, 54.92535197)>)>

offset_coord = <SkyCoord (SkyOffsetICRS: rotation=0.0 deg, origin=<ICRS Coordinate: (ra, dec) in deg
    (200.98141867, 54.92535197)>): (lon, lat) in deg
    (-0.03092788, 0.01243293)>

new_coord_ = <SkyCoord (ICRS): (ra, dec) in deg
    (200.92758103, 54.93777301)>
Using SkyOffsetFrame for Star Mizar 
User inputs: offset_arcmin = 2, camera_rotation_deg = -21.9

Original RA(Hr)/Dec: 13.3988, 54.9254
Original RA(Deg)/Dec: 200.9814, 54.9254
New  RA(Hr)/Dec:  13.3952, 54.9378
New  RA(Deg)/Dec:  200.9276, 54.9378
Delta RA/DEC(min): 0.0538,           -0.0124


### Update Properties for Star based on wiki query results above 

In [52]:
star_magnitude = 2.04;  target_alt_name = "HD 116656" # FIX THIS LINE AND BELOW BASED ON SEARCH
star_type = 'A2'
exposure = compute_exposure_time(star_magnitude)
print(f"Required exposure for mag {star_magnitude}: {exposure:.2f} s")

Required exposure for mag 2.04: 3.64 s


In [53]:
df.loc[ridx]=[f'{target_name}_{target_alt_name}_Typ_{star_type}',f'{target_alt_name}',f'{new_coord.ra.deg:.4f}',f'{new_coord.dec.deg:.4f}',
           f'{star_magnitude}',f'{exposure:.2f}','NA','NA','1','0','',f'{orig_coord.ra.hour:.4f}',f'{orig_coord.ra.deg:.4f}',f'{orig_coord.dec.deg:.4f}']
ridx += 1
df['Name1*'] = df['Name1*'].str.replace(' ', '_'); print(df[['Name1*','Name2*',"RA2000*","D2000*","Pmag~","Exp~"]])

                        Name1*        Name2*   RA2000*   D2000*  Pmag~  \
0    3C273_Quasar_3C273_Typ_Qu  Quasar_3C273  187.2470   2.0648   12.9   
1  HD_108959_HD108959_Typ_F0IV      HD108959  187.7800   1.3394      8   
2  Rasalhague_HD_159561_Typ_A5     HD 159561  263.7019  12.5725   2.07   
3  Rasalgethi_HD_156014_Typ_M5     HD 156014  258.6300  14.4028    5.3   
4      Altair_HD_187642_Typ_A7     HD 187642  297.6645   8.8808   0.76   
5        Vega_HD_172167_Typ_A0     HD 172167  279.1951  38.7961  0.026   
6        Dubhe_HD_95689_Typ_K0      HD 95689  165.8666  61.7635   1.79   
7      Scheat_HD_217906_Typ_M2     HD 217906  345.9085  28.0952   2.42   
8       Mizar_HD_116656_Typ_A2     HD 116656  200.9276  54.9378   2.04   

       Exp~  
0  52772.22  
1    699.58  
2      3.74  
3     64.60  
4      1.18  
5      0.62  
6      2.92  
7      5.09  
8      3.64  


# CELESTIAL COORDINATES FOR NEW STAR Alcor

In [54]:
# CELESTIAL COORDINATES FOR NEW STAR

from IPython.display import Markdown as md
# Instead of setting the cell to Markdown, create Markdown from withnin a code cell!
# We can just use python variable replacement syntax to make the text dynamic
target_name = "Alcor"
md(f"### Obtain spectral details for new star {target_name}")


### Obtain spectral details for new star Alcor

In [55]:
new_coord, orig_coord = obtain_info_for_star(target_name)

<SkyCoord (ICRS): (ra, dec) in deg
    (201.30640764, 54.98795966)>

sky_PA: 291.9 deg

offset_dist: 2.0 arcmin

dx: -1.8556725077978407 arcmin dy: 0.7459755651516162 arcmin

offset_frame = <SkyOffsetICRS Frame (rotation=0.0 deg, origin=<ICRS Coordinate: (ra, dec) in deg
    (201.30640764, 54.98795966)>)>

offset_coord = <SkyCoord (SkyOffsetICRS: rotation=0.0 deg, origin=<ICRS Coordinate: (ra, dec) in deg
    (201.30640764, 54.98795966)>): (lon, lat) in deg
    (-0.03092788, 0.01243293)>

new_coord_ = <SkyCoord (ICRS): (ra, dec) in deg
    (201.25248602, 55.00038067)>
Using SkyOffsetFrame for Star Alcor 
User inputs: offset_arcmin = 2, camera_rotation_deg = -21.9

Original RA(Hr)/Dec: 13.4204, 54.9880
Original RA(Deg)/Dec: 201.3064, 54.9880
New  RA(Hr)/Dec:  13.4168, 55.0004
New  RA(Deg)/Dec:  201.2525, 55.0004
Delta RA/DEC(min): 0.0539,           -0.0124


### Update Properties for Star based on wiki query results above 

In [56]:
star_magnitude = 3.88;  target_alt_name = "HD 116657" # FIX THIS LINE AND BELOW BASED ON SEARCH
star_type = 'MK'
exposure = compute_exposure_time(star_magnitude)
print(f"Required exposure for mag {star_magnitude}: {exposure:.2f} s")

Required exposure for mag 3.88: 18.46 s


In [57]:
df.loc[ridx]=[f'{target_name}_{target_alt_name}_Typ_{star_type}',f'{target_alt_name}',f'{new_coord.ra.deg:.4f}',f'{new_coord.dec.deg:.4f}',
           f'{star_magnitude}',f'{exposure:.2f}','NA','NA','1','0','',f'{orig_coord.ra.hour:.4f}',f'{orig_coord.ra.deg:.4f}',f'{orig_coord.dec.deg:.4f}']
ridx += 1
df['Name1*'] = df['Name1*'].str.replace(' ', '_'); print(df[['Name1*','Name2*',"RA2000*","D2000*","Pmag~","Exp~"]])

                        Name1*        Name2*   RA2000*   D2000*  Pmag~  \
0    3C273_Quasar_3C273_Typ_Qu  Quasar_3C273  187.2470   2.0648   12.9   
1  HD_108959_HD108959_Typ_F0IV      HD108959  187.7800   1.3394      8   
2  Rasalhague_HD_159561_Typ_A5     HD 159561  263.7019  12.5725   2.07   
3  Rasalgethi_HD_156014_Typ_M5     HD 156014  258.6300  14.4028    5.3   
4      Altair_HD_187642_Typ_A7     HD 187642  297.6645   8.8808   0.76   
5        Vega_HD_172167_Typ_A0     HD 172167  279.1951  38.7961  0.026   
6        Dubhe_HD_95689_Typ_K0      HD 95689  165.8666  61.7635   1.79   
7      Scheat_HD_217906_Typ_M2     HD 217906  345.9085  28.0952   2.42   
8       Mizar_HD_116656_Typ_A2     HD 116656  200.9276  54.9378   2.04   
9       Alcor_HD_116657_Typ_MK     HD 116657  201.2525  55.0004   3.88   

       Exp~  
0  52772.22  
1    699.58  
2      3.74  
3     64.60  
4      1.18  
5      0.62  
6      2.92  
7      5.09  
8      3.64  
9     18.46  


# CELESTIAL COORDINATES FOR NEW STAR R Lyr

In [58]:
# CELESTIAL COORDINATES FOR NEW STAR

from IPython.display import Markdown as md
# Instead of setting the cell to Markdown, create Markdown from withnin a code cell!
# We can just use python variable replacement syntax to make the text dynamic
target_name = "R Lyr"
md(f"### Obtain spectral details for new star {target_name}")


### Obtain spectral details for new star R Lyr

In [59]:
new_coord, orig_coord = obtain_info_for_star(target_name)

<SkyCoord (ICRS): (ra, dec) in deg
    (283.83375974, 43.94608958)>

sky_PA: 291.9 deg

offset_dist: 2.0 arcmin

dx: -1.8556725077978407 arcmin dy: 0.7459755651516162 arcmin

offset_frame = <SkyOffsetICRS Frame (rotation=0.0 deg, origin=<ICRS Coordinate: (ra, dec) in deg
    (283.83375974, 43.94608958)>)>

offset_coord = <SkyCoord (SkyOffsetICRS: rotation=0.0 deg, origin=<ICRS Coordinate: (ra, dec) in deg
    (283.83375974, 43.94608958)>): (lon, lat) in deg
    (-0.03092788, 0.01243293)>

new_coord_ = <SkyCoord (ICRS): (ra, dec) in deg
    (283.79079496, 43.95851446)>
Using SkyOffsetFrame for Star R Lyr 
User inputs: offset_arcmin = 2, camera_rotation_deg = -21.9

Original RA(Hr)/Dec: 18.9223, 43.9461
Original RA(Deg)/Dec: 283.8338, 43.9461
New  RA(Hr)/Dec:  18.9194, 43.9585
New  RA(Deg)/Dec:  283.7908, 43.9585
Delta RA/DEC(min): 0.0430,           -0.0124


### Update Properties for Star based on wiki query results above 

In [60]:
star_magnitude = 3.9;  target_alt_name = "HD 175865" # FIX THIS LINE AND BELOW BASED ON SEARCH
star_type = 'M5'
exposure = compute_exposure_time(star_magnitude)
print(f"Required exposure for mag {star_magnitude}: {exposure:.2f} s")

Required exposure for mag 3.9: 18.79 s


In [61]:
df.loc[ridx]=[f'{target_name}_{target_alt_name}_Typ_{star_type}',f'{target_alt_name}',f'{new_coord.ra.deg:.4f}',f'{new_coord.dec.deg:.4f}',
           f'{star_magnitude}',f'{exposure:.2f}','NA','NA','1','0','',f'{orig_coord.ra.hour:.4f}',f'{orig_coord.ra.deg:.4f}',f'{orig_coord.dec.deg:.4f}']
ridx += 1
df['Name1*'] = df['Name1*'].str.replace(' ', '_'); print(df[['Name1*','Name2*',"RA2000*","D2000*","Pmag~","Exp~"]])

                         Name1*        Name2*   RA2000*   D2000*  Pmag~  \
0     3C273_Quasar_3C273_Typ_Qu  Quasar_3C273  187.2470   2.0648   12.9   
1   HD_108959_HD108959_Typ_F0IV      HD108959  187.7800   1.3394      8   
2   Rasalhague_HD_159561_Typ_A5     HD 159561  263.7019  12.5725   2.07   
3   Rasalgethi_HD_156014_Typ_M5     HD 156014  258.6300  14.4028    5.3   
4       Altair_HD_187642_Typ_A7     HD 187642  297.6645   8.8808   0.76   
5         Vega_HD_172167_Typ_A0     HD 172167  279.1951  38.7961  0.026   
6         Dubhe_HD_95689_Typ_K0      HD 95689  165.8666  61.7635   1.79   
7       Scheat_HD_217906_Typ_M2     HD 217906  345.9085  28.0952   2.42   
8        Mizar_HD_116656_Typ_A2     HD 116656  200.9276  54.9378   2.04   
9        Alcor_HD_116657_Typ_MK     HD 116657  201.2525  55.0004   3.88   
10       R_Lyr_HD_175865_Typ_M5     HD 175865  283.7908  43.9585    3.9   

        Exp~  
0   52772.22  
1     699.58  
2       3.74  
3      64.60  
4       1.18  
5       0

# CELESTIAL COORDINATES FOR NEW STAR Alpheratz

In [62]:
# CELESTIAL COORDINATES FOR NEW STAR

from IPython.display import Markdown as md
# Instead of setting the cell to Markdown, create Markdown from withnin a code cell!
# We can just use python variable replacement syntax to make the text dynamic
target_name = "Alpheratz"
md(f"### Obtain spectral details for new star {target_name}")


### Obtain spectral details for new star Alpheratz

In [63]:
new_coord, orig_coord = obtain_info_for_star(target_name)

<SkyCoord (ICRS): (ra, dec) in deg
    (2.09691619, 29.09043112)>

sky_PA: 291.9 deg

offset_dist: 2.0 arcmin

dx: -1.8556725077978407 arcmin dy: 0.7459755651516162 arcmin

offset_frame = <SkyOffsetICRS Frame (rotation=0.0 deg, origin=<ICRS Coordinate: (ra, dec) in deg
    (2.09691619, 29.09043112)>)>

offset_coord = <SkyCoord (SkyOffsetICRS: rotation=0.0 deg, origin=<ICRS Coordinate: (ra, dec) in deg
    (2.09691619, 29.09043112)>): (lon, lat) in deg
    (-0.03092788, 0.01243293)>

new_coord_ = <SkyCoord (ICRS): (ra, dec) in deg
    (2.06151939, 29.1028594)>
Using SkyOffsetFrame for Star Alpheratz 
User inputs: offset_arcmin = 2, camera_rotation_deg = -21.9

Original RA(Hr)/Dec: 0.1398, 29.0904
Original RA(Deg)/Dec: 2.0969, 29.0904
New  RA(Hr)/Dec:  0.1374, 29.1029
New  RA(Deg)/Dec:  2.0615, 29.1029
Delta RA/DEC(min): 0.0354,           -0.0124


### Update Properties for Star based on wiki query results above 

In [64]:
star_magnitude = 2.06;  target_alt_name = "HD 358" # FIX THIS LINE AND BELOW BASED ON SEARCH
star_type = 'B8_A7'
exposure = compute_exposure_time(star_magnitude)
print(f"Required exposure for mag {star_magnitude}: {exposure:.2f} s")

Required exposure for mag 2.06: 3.70 s


In [65]:
df.loc[ridx]=[f'{target_name}_{target_alt_name}_Typ_{star_type}',f'{target_alt_name}',f'{new_coord.ra.deg:.4f}',f'{new_coord.dec.deg:.4f}',
           f'{star_magnitude}',f'{exposure:.2f}','NA','NA','1','0','',f'{orig_coord.ra.hour:.4f}',f'{orig_coord.ra.deg:.4f}',f'{orig_coord.dec.deg:.4f}']
ridx += 1
df['Name1*'] = df['Name1*'].str.replace(' ', '_'); print(df[['Name1*','Name2*',"RA2000*","D2000*","Pmag~","Exp~"]])

                         Name1*        Name2*   RA2000*   D2000*  Pmag~  \
0     3C273_Quasar_3C273_Typ_Qu  Quasar_3C273  187.2470   2.0648   12.9   
1   HD_108959_HD108959_Typ_F0IV      HD108959  187.7800   1.3394      8   
2   Rasalhague_HD_159561_Typ_A5     HD 159561  263.7019  12.5725   2.07   
3   Rasalgethi_HD_156014_Typ_M5     HD 156014  258.6300  14.4028    5.3   
4       Altair_HD_187642_Typ_A7     HD 187642  297.6645   8.8808   0.76   
5         Vega_HD_172167_Typ_A0     HD 172167  279.1951  38.7961  0.026   
6         Dubhe_HD_95689_Typ_K0      HD 95689  165.8666  61.7635   1.79   
7       Scheat_HD_217906_Typ_M2     HD 217906  345.9085  28.0952   2.42   
8        Mizar_HD_116656_Typ_A2     HD 116656  200.9276  54.9378   2.04   
9        Alcor_HD_116657_Typ_MK     HD 116657  201.2525  55.0004   3.88   
10       R_Lyr_HD_175865_Typ_M5     HD 175865  283.7908  43.9585    3.9   
11   Alpheratz_HD_358_Typ_B8_A7        HD 358    2.0615  29.1029   2.06   

        Exp~  
0   52772

# CELESTIAL COORDINATES FOR NEW STAR Albireo A

In [66]:
# CELESTIAL COORDINATES FOR NEW STAR

from IPython.display import Markdown as md
# Instead of setting the cell to Markdown, create Markdown from withnin a code cell!
# We can just use python variable replacement syntax to make the text dynamic
target_name = "Albireo"
md(f"### Obtain spectral details for new star {target_name}")


### Obtain spectral details for new star Albireo

In [67]:
new_coord, orig_coord = obtain_info_for_star(target_name)

<SkyCoord (ICRS): (ra, dec) in deg
    (292.68031501, 27.95967363)>

sky_PA: 291.9 deg

offset_dist: 2.0 arcmin

dx: -1.8556725077978407 arcmin dy: 0.7459755651516162 arcmin

offset_frame = <SkyOffsetICRS Frame (rotation=0.0 deg, origin=<ICRS Coordinate: (ra, dec) in deg
    (292.68031501, 27.95967363)>)>

offset_coord = <SkyCoord (SkyOffsetICRS: rotation=0.0 deg, origin=<ICRS Coordinate: (ra, dec) in deg
    (292.68031501, 27.95967363)>): (lon, lat) in deg
    (-0.03092788, 0.01243293)>

new_coord_ = <SkyCoord (ICRS): (ra, dec) in deg
    (292.64529609, 27.97210213)>
Using SkyOffsetFrame for Star Albireo 
User inputs: offset_arcmin = 2, camera_rotation_deg = -21.9

Original RA(Hr)/Dec: 19.5120, 27.9597
Original RA(Deg)/Dec: 292.6803, 27.9597
New  RA(Hr)/Dec:  19.5097, 27.9721
New  RA(Deg)/Dec:  292.6453, 27.9721
Delta RA/DEC(min): 0.0350,           -0.0124


### Update Properties for Star based on wiki query results above 

In [68]:
star_magnitude = 3.21;  target_alt_name = "HD 183912" # FIX THIS LINE AND BELOW BASED ON SEARCH
star_type = 'K2'
exposure = compute_exposure_time(star_magnitude)
print(f"Required exposure for mag {star_magnitude}: {exposure:.2f} s")

Required exposure for mag 3.21: 10.22 s


In [69]:
df.loc[ridx]=[f'{target_name}_{target_alt_name}_Typ_{star_type}',f'{target_alt_name}',f'{new_coord.ra.deg:.4f}',f'{new_coord.dec.deg:.4f}',
           f'{star_magnitude}',f'{exposure:.2f}','NA','NA','1','0','',f'{orig_coord.ra.hour:.4f}',f'{orig_coord.ra.deg:.4f}',f'{orig_coord.dec.deg:.4f}']
ridx += 1
df['Name1*'] = df['Name1*'].str.replace(' ', '_'); print(df[['Name1*','Name2*',"RA2000*","D2000*","Pmag~","Exp~"]])

                         Name1*        Name2*   RA2000*   D2000*  Pmag~  \
0     3C273_Quasar_3C273_Typ_Qu  Quasar_3C273  187.2470   2.0648   12.9   
1   HD_108959_HD108959_Typ_F0IV      HD108959  187.7800   1.3394      8   
2   Rasalhague_HD_159561_Typ_A5     HD 159561  263.7019  12.5725   2.07   
3   Rasalgethi_HD_156014_Typ_M5     HD 156014  258.6300  14.4028    5.3   
4       Altair_HD_187642_Typ_A7     HD 187642  297.6645   8.8808   0.76   
5         Vega_HD_172167_Typ_A0     HD 172167  279.1951  38.7961  0.026   
6         Dubhe_HD_95689_Typ_K0      HD 95689  165.8666  61.7635   1.79   
7       Scheat_HD_217906_Typ_M2     HD 217906  345.9085  28.0952   2.42   
8        Mizar_HD_116656_Typ_A2     HD 116656  200.9276  54.9378   2.04   
9        Alcor_HD_116657_Typ_MK     HD 116657  201.2525  55.0004   3.88   
10       R_Lyr_HD_175865_Typ_M5     HD 175865  283.7908  43.9585    3.9   
11   Alpheratz_HD_358_Typ_B8_A7        HD 358    2.0615  29.1029   2.06   
12     Albireo_HD_183912_

# CELESTIAL COORDINATES FOR NEW STAR Albireo B

In [70]:
# CELESTIAL COORDINATES FOR NEW STAR

from IPython.display import Markdown as md
# Instead of setting the cell to Markdown, create Markdown from withnin a code cell!
# We can just use python variable replacement syntax to make the text dynamic
target_name = "Albireo"
md(f"### Obtain spectral details for new star {target_name}")


### Obtain spectral details for new star Albireo

In [71]:
new_coord, orig_coord = obtain_info_for_star(target_name)

<SkyCoord (ICRS): (ra, dec) in deg
    (292.68031501, 27.95967363)>

sky_PA: 291.9 deg

offset_dist: 2.0 arcmin

dx: -1.8556725077978407 arcmin dy: 0.7459755651516162 arcmin

offset_frame = <SkyOffsetICRS Frame (rotation=0.0 deg, origin=<ICRS Coordinate: (ra, dec) in deg
    (292.68031501, 27.95967363)>)>

offset_coord = <SkyCoord (SkyOffsetICRS: rotation=0.0 deg, origin=<ICRS Coordinate: (ra, dec) in deg
    (292.68031501, 27.95967363)>): (lon, lat) in deg
    (-0.03092788, 0.01243293)>

new_coord_ = <SkyCoord (ICRS): (ra, dec) in deg
    (292.64529609, 27.97210213)>
Using SkyOffsetFrame for Star Albireo 
User inputs: offset_arcmin = 2, camera_rotation_deg = -21.9

Original RA(Hr)/Dec: 19.5120, 27.9597
Original RA(Deg)/Dec: 292.6803, 27.9597
New  RA(Hr)/Dec:  19.5097, 27.9721
New  RA(Deg)/Dec:  292.6453, 27.9721
Delta RA/DEC(min): 0.0350,           -0.0124


### Update Properties for Star based on wiki query results above 

In [72]:
star_magnitude = 5.11;  target_alt_name = "HD 183913" # FIX THIS LINE AND BELOW BASED ON SEARCH
star_type = 'B8'
exposure = compute_exposure_time(star_magnitude)
print(f"Required exposure for mag {star_magnitude}: {exposure:.2f} s")

Required exposure for mag 5.11: 54.63 s


In [73]:
df.loc[ridx]=[f'{target_name}_{target_alt_name}_Typ_{star_type}',f'{target_alt_name}',f'{new_coord.ra.deg:.4f}',f'{new_coord.dec.deg:.4f}',
           f'{star_magnitude}',f'{exposure:.2f}','NA','NA','1','0','',f'{orig_coord.ra.hour:.4f}',f'{orig_coord.ra.deg:.4f}',f'{orig_coord.dec.deg:.4f}']
ridx += 1
df['Name1*'] = df['Name1*'].str.replace(' ', '_'); print(df[['Name1*','Name2*',"RA2000*","D2000*","Pmag~","Exp~"]])

                         Name1*        Name2*   RA2000*   D2000*  Pmag~  \
0     3C273_Quasar_3C273_Typ_Qu  Quasar_3C273  187.2470   2.0648   12.9   
1   HD_108959_HD108959_Typ_F0IV      HD108959  187.7800   1.3394      8   
2   Rasalhague_HD_159561_Typ_A5     HD 159561  263.7019  12.5725   2.07   
3   Rasalgethi_HD_156014_Typ_M5     HD 156014  258.6300  14.4028    5.3   
4       Altair_HD_187642_Typ_A7     HD 187642  297.6645   8.8808   0.76   
5         Vega_HD_172167_Typ_A0     HD 172167  279.1951  38.7961  0.026   
6         Dubhe_HD_95689_Typ_K0      HD 95689  165.8666  61.7635   1.79   
7       Scheat_HD_217906_Typ_M2     HD 217906  345.9085  28.0952   2.42   
8        Mizar_HD_116656_Typ_A2     HD 116656  200.9276  54.9378   2.04   
9        Alcor_HD_116657_Typ_MK     HD 116657  201.2525  55.0004   3.88   
10       R_Lyr_HD_175865_Typ_M5     HD 175865  283.7908  43.9585    3.9   
11   Alpheratz_HD_358_Typ_B8_A7        HD 358    2.0615  29.1029   2.06   
12     Albireo_HD_183912_

# CELESTIAL COORDINATES FOR NEW STAR Denebola

In [74]:
# CELESTIAL COORDINATES FOR NEW STAR

from IPython.display import Markdown as md
# Instead of setting the cell to Markdown, create Markdown from withnin a code cell!
# We can just use python variable replacement syntax to make the text dynamic
target_name = "Denebola"
md(f"### Obtain spectral details for new star {target_name}")


### Obtain spectral details for new star Denebola

In [75]:
new_coord, orig_coord = obtain_info_for_star(target_name)

<SkyCoord (ICRS): (ra, dec) in deg
    (177.26490976, 14.57205807)>

sky_PA: 291.9 deg

offset_dist: 2.0 arcmin

dx: -1.8556725077978407 arcmin dy: 0.7459755651516162 arcmin

offset_frame = <SkyOffsetICRS Frame (rotation=0.0 deg, origin=<ICRS Coordinate: (ra, dec) in deg
    (177.26490976, 14.57205807)>)>

offset_coord = <SkyCoord (SkyOffsetICRS: rotation=0.0 deg, origin=<ICRS Coordinate: (ra, dec) in deg
    (177.26490976, 14.57205807)>): (lon, lat) in deg
    (-0.03092788, 0.01243293)>

new_coord_ = <SkyCoord (ICRS): (ra, dec) in deg
    (177.23295212, 14.58448882)>
Using SkyOffsetFrame for Star Denebola 
User inputs: offset_arcmin = 2, camera_rotation_deg = -21.9

Original RA(Hr)/Dec: 11.8177, 14.5721
Original RA(Deg)/Dec: 177.2649, 14.5721
New  RA(Hr)/Dec:  11.8155, 14.5845
New  RA(Deg)/Dec:  177.2330, 14.5845
Delta RA/DEC(min): 0.0320,           -0.0124


### Update Properties for Star based on wiki query results above 

In [76]:
star_magnitude = 2.14;  target_alt_name = "HD 102647" # FIX THIS LINE AND BELOW BASED ON SEARCH
star_type = 'A3'
exposure = compute_exposure_time(star_magnitude)
print(f"Required exposure for mag {star_magnitude}: {exposure:.2f} s")

Required exposure for mag 2.14: 3.98 s


In [77]:
df.loc[ridx]=[f'{target_name}_{target_alt_name}_Typ_{star_type}',f'{target_alt_name}',f'{new_coord.ra.deg:.4f}',f'{new_coord.dec.deg:.4f}',
           f'{star_magnitude}',f'{exposure:.2f}','NA','NA','1','0','',f'{orig_coord.ra.hour:.4f}',f'{orig_coord.ra.deg:.4f}',f'{orig_coord.dec.deg:.4f}']
ridx += 1
df['Name1*'] = df['Name1*'].str.replace(' ', '_'); print(df[['Name1*','Name2*',"RA2000*","D2000*","Pmag~","Exp~"]])

                         Name1*        Name2*   RA2000*   D2000*  Pmag~  \
0     3C273_Quasar_3C273_Typ_Qu  Quasar_3C273  187.2470   2.0648   12.9   
1   HD_108959_HD108959_Typ_F0IV      HD108959  187.7800   1.3394      8   
2   Rasalhague_HD_159561_Typ_A5     HD 159561  263.7019  12.5725   2.07   
3   Rasalgethi_HD_156014_Typ_M5     HD 156014  258.6300  14.4028    5.3   
4       Altair_HD_187642_Typ_A7     HD 187642  297.6645   8.8808   0.76   
5         Vega_HD_172167_Typ_A0     HD 172167  279.1951  38.7961  0.026   
6         Dubhe_HD_95689_Typ_K0      HD 95689  165.8666  61.7635   1.79   
7       Scheat_HD_217906_Typ_M2     HD 217906  345.9085  28.0952   2.42   
8        Mizar_HD_116656_Typ_A2     HD 116656  200.9276  54.9378   2.04   
9        Alcor_HD_116657_Typ_MK     HD 116657  201.2525  55.0004   3.88   
10       R_Lyr_HD_175865_Typ_M5     HD 175865  283.7908  43.9585    3.9   
11   Alpheratz_HD_358_Typ_B8_A7        HD 358    2.0615  29.1029   2.06   
12     Albireo_HD_183912_

# CELESTIAL COORDINATES FOR NEW STAR Zosma

In [78]:
# CELESTIAL COORDINATES FOR NEW STAR

from IPython.display import Markdown as md
# Instead of setting the cell to Markdown, create Markdown from withnin a code cell!
# We can just use python variable replacement syntax to make the text dynamic
target_name = "Zosma"
md(f"### Obtain spectral details for new star {target_name}")


### Obtain spectral details for new star Zosma

In [79]:
new_coord, orig_coord = obtain_info_for_star(target_name)

<SkyCoord (ICRS): (ra, dec) in deg
    (168.52708927, 20.52371814)>

sky_PA: 291.9 deg

offset_dist: 2.0 arcmin

dx: -1.8556725077978407 arcmin dy: 0.7459755651516162 arcmin

offset_frame = <SkyOffsetICRS Frame (rotation=0.0 deg, origin=<ICRS Coordinate: (ra, dec) in deg
    (168.52708927, 20.52371814)>)>

offset_coord = <SkyCoord (SkyOffsetICRS: rotation=0.0 deg, origin=<ICRS Coordinate: (ra, dec) in deg
    (168.52708927, 20.52371814)>): (lon, lat) in deg
    (-0.03092788, 0.01243293)>

new_coord_ = <SkyCoord (ICRS): (ra, dec) in deg
    (168.49406258, 20.53614794)>
Using SkyOffsetFrame for Star Zosma 
User inputs: offset_arcmin = 2, camera_rotation_deg = -21.9

Original RA(Hr)/Dec: 11.2351, 20.5237
Original RA(Deg)/Dec: 168.5271, 20.5237
New  RA(Hr)/Dec:  11.2329, 20.5361
New  RA(Deg)/Dec:  168.4941, 20.5361
Delta RA/DEC(min): 0.0330,           -0.0124


### Update Properties for Star based on wiki query results above 

In [80]:
star_magnitude = 2.56;  target_alt_name = "HD 97603" # FIX THIS LINE AND BELOW BASED ON SEARCH
star_type = 'A4'
exposure = compute_exposure_time(star_magnitude)
print(f"Required exposure for mag {star_magnitude}: {exposure:.2f} s")

Required exposure for mag 2.56: 5.76 s


In [81]:
df.loc[ridx]=[f'{target_name}_{target_alt_name}_Typ_{star_type}',f'{target_alt_name}',f'{new_coord.ra.deg:.4f}',f'{new_coord.dec.deg:.4f}',
           f'{star_magnitude}',f'{exposure:.2f}','NA','NA','1','0','',f'{orig_coord.ra.hour:.4f}',f'{orig_coord.ra.deg:.4f}',f'{orig_coord.dec.deg:.4f}']
ridx += 1
df['Name1*'] = df['Name1*'].str.replace(' ', '_'); print(df[['Name1*','Name2*',"RA2000*","D2000*","Pmag~","Exp~"]])

                         Name1*        Name2*   RA2000*   D2000*  Pmag~  \
0     3C273_Quasar_3C273_Typ_Qu  Quasar_3C273  187.2470   2.0648   12.9   
1   HD_108959_HD108959_Typ_F0IV      HD108959  187.7800   1.3394      8   
2   Rasalhague_HD_159561_Typ_A5     HD 159561  263.7019  12.5725   2.07   
3   Rasalgethi_HD_156014_Typ_M5     HD 156014  258.6300  14.4028    5.3   
4       Altair_HD_187642_Typ_A7     HD 187642  297.6645   8.8808   0.76   
5         Vega_HD_172167_Typ_A0     HD 172167  279.1951  38.7961  0.026   
6         Dubhe_HD_95689_Typ_K0      HD 95689  165.8666  61.7635   1.79   
7       Scheat_HD_217906_Typ_M2     HD 217906  345.9085  28.0952   2.42   
8        Mizar_HD_116656_Typ_A2     HD 116656  200.9276  54.9378   2.04   
9        Alcor_HD_116657_Typ_MK     HD 116657  201.2525  55.0004   3.88   
10       R_Lyr_HD_175865_Typ_M5     HD 175865  283.7908  43.9585    3.9   
11   Alpheratz_HD_358_Typ_B8_A7        HD 358    2.0615  29.1029   2.06   
12     Albireo_HD_183912_

# CELESTIAL COORDINATES FOR NEW STAR Alioth

In [82]:
# CELESTIAL COORDINATES FOR NEW STAR

from IPython.display import Markdown as md
# Instead of setting the cell to Markdown, create Markdown from withnin a code cell!
# We can just use python variable replacement syntax to make the text dynamic
target_name = "Alioth"
md(f"### Obtain spectral details for new star {target_name}")


### Obtain spectral details for new star Alioth

In [83]:
new_coord, orig_coord = obtain_info_for_star(target_name)

<SkyCoord (ICRS): (ra, dec) in deg
    (193.50728997, 55.95982296)>

sky_PA: 291.9 deg

offset_dist: 2.0 arcmin

dx: -1.8556725077978407 arcmin dy: 0.7459755651516162 arcmin

offset_frame = <SkyOffsetICRS Frame (rotation=0.0 deg, origin=<ICRS Coordinate: (ra, dec) in deg
    (193.50728997, 55.95982296)>)>

offset_coord = <SkyCoord (SkyOffsetICRS: rotation=0.0 deg, origin=<ICRS Coordinate: (ra, dec) in deg
    (193.50728997, 55.95982296)>): (lon, lat) in deg
    (-0.03092788, 0.01243293)>

new_coord_ = <SkyCoord (ICRS): (ra, dec) in deg
    (193.45202159, 55.97224352)>
Using SkyOffsetFrame for Star Alioth 
User inputs: offset_arcmin = 2, camera_rotation_deg = -21.9

Original RA(Hr)/Dec: 12.9005, 55.9598
Original RA(Deg)/Dec: 193.5073, 55.9598
New  RA(Hr)/Dec:  12.8968, 55.9722
New  RA(Deg)/Dec:  193.4520, 55.9722
Delta RA/DEC(min): 0.0553,           -0.0124


### Update Properties for Star based on wiki query results above 

In [84]:
star_magnitude = 1.77;  target_alt_name = "HD 112185" # FIX THIS LINE AND BELOW BASED ON SEARCH
star_type = 'A1'
exposure = compute_exposure_time(star_magnitude)
print(f"Required exposure for mag {star_magnitude}: {exposure:.2f} s")

Required exposure for mag 1.77: 2.87 s


In [85]:
df.loc[ridx]=[f'{target_name}_{target_alt_name}_Typ_{star_type}',f'{target_alt_name}',f'{new_coord.ra.deg:.4f}',f'{new_coord.dec.deg:.4f}',
           f'{star_magnitude}',f'{exposure:.2f}','NA','NA','1','0','',f'{orig_coord.ra.hour:.4f}',f'{orig_coord.ra.deg:.4f}',f'{orig_coord.dec.deg:.4f}']
ridx += 1
df['Name1*'] = df['Name1*'].str.replace(' ', '_'); print(df[['Name1*','Name2*',"RA2000*","D2000*","Pmag~","Exp~"]])

                         Name1*        Name2*   RA2000*   D2000*  Pmag~  \
0     3C273_Quasar_3C273_Typ_Qu  Quasar_3C273  187.2470   2.0648   12.9   
1   HD_108959_HD108959_Typ_F0IV      HD108959  187.7800   1.3394      8   
2   Rasalhague_HD_159561_Typ_A5     HD 159561  263.7019  12.5725   2.07   
3   Rasalgethi_HD_156014_Typ_M5     HD 156014  258.6300  14.4028    5.3   
4       Altair_HD_187642_Typ_A7     HD 187642  297.6645   8.8808   0.76   
5         Vega_HD_172167_Typ_A0     HD 172167  279.1951  38.7961  0.026   
6         Dubhe_HD_95689_Typ_K0      HD 95689  165.8666  61.7635   1.79   
7       Scheat_HD_217906_Typ_M2     HD 217906  345.9085  28.0952   2.42   
8        Mizar_HD_116656_Typ_A2     HD 116656  200.9276  54.9378   2.04   
9        Alcor_HD_116657_Typ_MK     HD 116657  201.2525  55.0004   3.88   
10       R_Lyr_HD_175865_Typ_M5     HD 175865  283.7908  43.9585    3.9   
11   Alpheratz_HD_358_Typ_B8_A7        HD 358    2.0615  29.1029   2.06   
12     Albireo_HD_183912_

# CELESTIAL COORDINATES FOR NEW STAR Minelauva

In [86]:
# CELESTIAL COORDINATES FOR NEW STAR

from IPython.display import Markdown as md
# Instead of setting the cell to Markdown, create Markdown from withnin a code cell!
# We can just use python variable replacement syntax to make the text dynamic
target_name = "Minelauva"
md(f"### Obtain spectral details for new star {target_name}")


### Obtain spectral details for new star Minelauva

In [87]:
new_coord, orig_coord = obtain_info_for_star(target_name)

<SkyCoord (ICRS): (ra, dec) in deg
    (193.90086927, 3.3974689)>

sky_PA: 291.9 deg

offset_dist: 2.0 arcmin

dx: -1.8556725077978407 arcmin dy: 0.7459755651516162 arcmin

offset_frame = <SkyOffsetICRS Frame (rotation=0.0 deg, origin=<ICRS Coordinate: (ra, dec) in deg
    (193.90086927, 3.3974689)>)>

offset_coord = <SkyCoord (SkyOffsetICRS: rotation=0.0 deg, origin=<ICRS Coordinate: (ra, dec) in deg
    (193.90086927, 3.3974689)>): (lon, lat) in deg
    (-0.03092788, 0.01243293)>

new_coord_ = <SkyCoord (ICRS): (ra, dec) in deg
    (193.86988654, 3.40990133)>
Using SkyOffsetFrame for Star Minelauva 
User inputs: offset_arcmin = 2, camera_rotation_deg = -21.9

Original RA(Hr)/Dec: 12.9267, 3.3975
Original RA(Deg)/Dec: 193.9009, 3.3975
New  RA(Hr)/Dec:  12.9247, 3.4099
New  RA(Deg)/Dec:  193.8699, 3.4099
Delta RA/DEC(min): 0.0310,           -0.0124


### Update Properties for Star based on wiki query results above 

In [88]:
star_magnitude = 3.32;  target_alt_name = "HD 112300" # FIX THIS LINE AND BELOW BASED ON SEARCH
star_type = 'M3'
exposure = compute_exposure_time(star_magnitude)
print(f"Required exposure for mag {star_magnitude}: {exposure:.2f} s")

Required exposure for mag 3.32: 11.26 s


In [89]:
df.loc[ridx]=[f'{target_name}_{target_alt_name}_Typ_{star_type}',f'{target_alt_name}',f'{new_coord.ra.deg:.4f}',f'{new_coord.dec.deg:.4f}',
           f'{star_magnitude}',f'{exposure:.2f}','NA','NA','1','0','',f'{orig_coord.ra.hour:.4f}',f'{orig_coord.ra.deg:.4f}',f'{orig_coord.dec.deg:.4f}']
ridx += 1
df['Name1*'] = df['Name1*'].str.replace(' ', '_'); print(df[['Name1*','Name2*',"RA2000*","D2000*","Pmag~","Exp~"]])

                         Name1*        Name2*   RA2000*   D2000*  Pmag~  \
0     3C273_Quasar_3C273_Typ_Qu  Quasar_3C273  187.2470   2.0648   12.9   
1   HD_108959_HD108959_Typ_F0IV      HD108959  187.7800   1.3394      8   
2   Rasalhague_HD_159561_Typ_A5     HD 159561  263.7019  12.5725   2.07   
3   Rasalgethi_HD_156014_Typ_M5     HD 156014  258.6300  14.4028    5.3   
4       Altair_HD_187642_Typ_A7     HD 187642  297.6645   8.8808   0.76   
5         Vega_HD_172167_Typ_A0     HD 172167  279.1951  38.7961  0.026   
6         Dubhe_HD_95689_Typ_K0      HD 95689  165.8666  61.7635   1.79   
7       Scheat_HD_217906_Typ_M2     HD 217906  345.9085  28.0952   2.42   
8        Mizar_HD_116656_Typ_A2     HD 116656  200.9276  54.9378   2.04   
9        Alcor_HD_116657_Typ_MK     HD 116657  201.2525  55.0004   3.88   
10       R_Lyr_HD_175865_Typ_M5     HD 175865  283.7908  43.9585    3.9   
11   Alpheratz_HD_358_Typ_B8_A7        HD 358    2.0615  29.1029   2.06   
12     Albireo_HD_183912_

# CELESTIAL COORDINATES FOR NEW STAR Arcturus

In [90]:
# CELESTIAL COORDINATES FOR NEW STAR

from IPython.display import Markdown as md
# Instead of setting the cell to Markdown, create Markdown from withnin a code cell!
# We can just use python variable replacement syntax to make the text dynamic
target_name = "Arcturus"
md(f"### Obtain spectral details for new star {target_name}")


### Obtain spectral details for new star Arcturus

In [91]:
new_coord, orig_coord = obtain_info_for_star(target_name)

<SkyCoord (ICRS): (ra, dec) in deg
    (213.9153003, 19.18240916)>

sky_PA: 291.9 deg

offset_dist: 2.0 arcmin

dx: -1.8556725077978407 arcmin dy: 0.7459755651516162 arcmin

offset_frame = <SkyOffsetICRS Frame (rotation=0.0 deg, origin=<ICRS Coordinate: (ra, dec) in deg
    (213.9153003, 19.18240916)>)>

offset_coord = <SkyCoord (SkyOffsetICRS: rotation=0.0 deg, origin=<ICRS Coordinate: (ra, dec) in deg
    (213.9153003, 19.18240916)>): (lon, lat) in deg
    (-0.03092788, 0.01243293)>

new_coord_ = <SkyCoord (ICRS): (ra, dec) in deg
    (213.8825518, 19.19483918)>
Using SkyOffsetFrame for Star Arcturus 
User inputs: offset_arcmin = 2, camera_rotation_deg = -21.9

Original RA(Hr)/Dec: 14.2610, 19.1824
Original RA(Deg)/Dec: 213.9153, 19.1824
New  RA(Hr)/Dec:  14.2588, 19.1948
New  RA(Deg)/Dec:  213.8826, 19.1948
Delta RA/DEC(min): 0.0327,           -0.0124


### Update Properties for Star based on wiki query results above 

In [92]:
star_magnitude = -0.05;  target_alt_name = "HD 124897" # FIX THIS LINE AND BELOW BASED ON SEARCH
star_type = 'K1'
exposure = compute_exposure_time(star_magnitude)
print(f"Required exposure for mag {star_magnitude}: {exposure:.2f} s")

Required exposure for mag -0.05: 0.58 s


In [93]:
df.loc[ridx]=[f'{target_name}_{target_alt_name}_Typ_{star_type}',f'{target_alt_name}',f'{new_coord.ra.deg:.4f}',f'{new_coord.dec.deg:.4f}',
           f'{star_magnitude}',f'{exposure:.2f}','NA','NA','1','0','',f'{orig_coord.ra.hour:.4f}',f'{orig_coord.ra.deg:.4f}',f'{orig_coord.dec.deg:.4f}']
ridx += 1
df['Name1*'] = df['Name1*'].str.replace(' ', '_'); print(df[['Name1*','Name2*',"RA2000*","D2000*","Pmag~","Exp~"]])

                         Name1*        Name2*   RA2000*   D2000*  Pmag~  \
0     3C273_Quasar_3C273_Typ_Qu  Quasar_3C273  187.2470   2.0648   12.9   
1   HD_108959_HD108959_Typ_F0IV      HD108959  187.7800   1.3394      8   
2   Rasalhague_HD_159561_Typ_A5     HD 159561  263.7019  12.5725   2.07   
3   Rasalgethi_HD_156014_Typ_M5     HD 156014  258.6300  14.4028    5.3   
4       Altair_HD_187642_Typ_A7     HD 187642  297.6645   8.8808   0.76   
5         Vega_HD_172167_Typ_A0     HD 172167  279.1951  38.7961  0.026   
6         Dubhe_HD_95689_Typ_K0      HD 95689  165.8666  61.7635   1.79   
7       Scheat_HD_217906_Typ_M2     HD 217906  345.9085  28.0952   2.42   
8        Mizar_HD_116656_Typ_A2     HD 116656  200.9276  54.9378   2.04   
9        Alcor_HD_116657_Typ_MK     HD 116657  201.2525  55.0004   3.88   
10       R_Lyr_HD_175865_Typ_M5     HD 175865  283.7908  43.9585    3.9   
11   Alpheratz_HD_358_Typ_B8_A7        HD 358    2.0615  29.1029   2.06   
12     Albireo_HD_183912_

# CELESTIAL COORDINATES FOR NEW STAR P Cyg

In [94]:
# CELESTIAL COORDINATES FOR NEW STAR

from IPython.display import Markdown as md
# Instead of setting the cell to Markdown, create Markdown from withnin a code cell!
# We can just use python variable replacement syntax to make the text dynamic
target_name = "P Cyg"
md(f"### Obtain spectral details for new star {target_name}")


### Obtain spectral details for new star P Cyg

In [95]:
new_coord, orig_coord = obtain_info_for_star(target_name)

<SkyCoord (ICRS): (ra, dec) in deg
    (304.44667489, 38.03293031)>

sky_PA: 291.9 deg

offset_dist: 2.0 arcmin

dx: -1.8556725077978407 arcmin dy: 0.7459755651516162 arcmin

offset_frame = <SkyOffsetICRS Frame (rotation=0.0 deg, origin=<ICRS Coordinate: (ra, dec) in deg
    (304.44667489, 38.03293031)>)>

offset_coord = <SkyCoord (SkyOffsetICRS: rotation=0.0 deg, origin=<ICRS Coordinate: (ra, dec) in deg
    (304.44667489, 38.03293031)>): (lon, lat) in deg
    (-0.03092788, 0.01243293)>

new_coord_ = <SkyCoord (ICRS): (ra, dec) in deg
    (304.40740255, 38.04535671)>
Using SkyOffsetFrame for Star P Cyg 
User inputs: offset_arcmin = 2, camera_rotation_deg = -21.9

Original RA(Hr)/Dec: 20.2964, 38.0329
Original RA(Deg)/Dec: 304.4467, 38.0329
New  RA(Hr)/Dec:  20.2938, 38.0454
New  RA(Deg)/Dec:  304.4074, 38.0454
Delta RA/DEC(min): 0.0393,           -0.0124


### Update Properties for Star based on wiki query results above 

In [96]:
star_magnitude = 4.82;  target_alt_name = "HD 193237" # FIX THIS LINE AND BELOW BASED ON SEARCH
star_type = 'B1'
exposure = compute_exposure_time(star_magnitude)
print(f"Required exposure for mag {star_magnitude}: {exposure:.2f} s")

Required exposure for mag 4.82: 42.30 s


In [97]:
df.loc[ridx]=[f'{target_name}_{target_alt_name}_Typ_{star_type}',f'{target_alt_name}',f'{new_coord.ra.deg:.4f}',f'{new_coord.dec.deg:.4f}',
           f'{star_magnitude}',f'{exposure:.2f}','NA','NA','1','0','',f'{orig_coord.ra.hour:.4f}',f'{orig_coord.ra.deg:.4f}',f'{orig_coord.dec.deg:.4f}']
ridx += 1
df['Name1*'] = df['Name1*'].str.replace(' ', '_'); print(df[['Name1*','Name2*',"RA2000*","D2000*","Pmag~","Exp~"]])

                         Name1*        Name2*   RA2000*   D2000*  Pmag~  \
0     3C273_Quasar_3C273_Typ_Qu  Quasar_3C273  187.2470   2.0648   12.9   
1   HD_108959_HD108959_Typ_F0IV      HD108959  187.7800   1.3394      8   
2   Rasalhague_HD_159561_Typ_A5     HD 159561  263.7019  12.5725   2.07   
3   Rasalgethi_HD_156014_Typ_M5     HD 156014  258.6300  14.4028    5.3   
4       Altair_HD_187642_Typ_A7     HD 187642  297.6645   8.8808   0.76   
5         Vega_HD_172167_Typ_A0     HD 172167  279.1951  38.7961  0.026   
6         Dubhe_HD_95689_Typ_K0      HD 95689  165.8666  61.7635   1.79   
7       Scheat_HD_217906_Typ_M2     HD 217906  345.9085  28.0952   2.42   
8        Mizar_HD_116656_Typ_A2     HD 116656  200.9276  54.9378   2.04   
9        Alcor_HD_116657_Typ_MK     HD 116657  201.2525  55.0004   3.88   
10       R_Lyr_HD_175865_Typ_M5     HD 175865  283.7908  43.9585    3.9   
11   Alpheratz_HD_358_Typ_B8_A7        HD 358    2.0615  29.1029   2.06   
12     Albireo_HD_183912_

# CELESTIAL COORDINATES FOR NEW STAR Polaris

In [98]:
# CELESTIAL COORDINATES FOR NEW STAR

from IPython.display import Markdown as md
# Instead of setting the cell to Markdown, create Markdown from withnin a code cell!
# We can just use python variable replacement syntax to make the text dynamic
target_name = "Polaris"
md(f"### Obtain spectral details for new star {target_name}")


### Obtain spectral details for new star Polaris

In [99]:
new_coord, orig_coord = obtain_info_for_star(target_name)

<SkyCoord (ICRS): (ra, dec) in deg
    (37.95456067, 89.26410897)>

sky_PA: 291.9 deg

offset_dist: 2.0 arcmin

dx: -1.8556725077978407 arcmin dy: 0.7459755651516162 arcmin

offset_frame = <SkyOffsetICRS Frame (rotation=0.0 deg, origin=<ICRS Coordinate: (ra, dec) in deg
    (37.95456067, 89.26410897)>)>

offset_coord = <SkyCoord (SkyOffsetICRS: rotation=0.0 deg, origin=<ICRS Coordinate: (ra, dec) in deg
    (37.95456067, 89.26410897)>): (lon, lat) in deg
    (-0.03092788, 0.01243293)>

new_coord_ = <SkyCoord (ICRS): (ra, dec) in deg
    (35.50658819, 89.27588115)>
Using SkyOffsetFrame for Star Polaris 
User inputs: offset_arcmin = 2, camera_rotation_deg = -21.9

Original RA(Hr)/Dec: 2.5303, 89.2641
Original RA(Deg)/Dec: 37.9546, 89.2641
New  RA(Hr)/Dec:  2.3671, 89.2759
New  RA(Deg)/Dec:  35.5066, 89.2759
Delta RA/DEC(min): 2.4480,           -0.0118


### Update Properties for Star based on wiki query results above 

In [100]:
star_magnitude = 1.98;  target_alt_name = "HD 8890" # FIX THIS LINE AND BELOW BASED ON SEARCH
star_type = 'F7'
exposure = compute_exposure_time(star_magnitude)
print(f"Required exposure for mag {star_magnitude}: {exposure:.2f} s")

Required exposure for mag 1.98: 3.45 s


In [101]:
df.loc[ridx]=[f'{target_name}_{target_alt_name}_Typ_{star_type}',f'{target_alt_name}',f'{new_coord.ra.deg:.4f}',f'{new_coord.dec.deg:.4f}',
           f'{star_magnitude}',f'{exposure:.2f}','NA','NA','1','0','',f'{orig_coord.ra.hour:.4f}',f'{orig_coord.ra.deg:.4f}',f'{orig_coord.dec.deg:.4f}']
ridx += 1
df['Name1*'] = df['Name1*'].str.replace(' ', '_'); print(df[['Name1*','Name2*',"RA2000*","D2000*","Pmag~","Exp~"]])

                         Name1*        Name2*   RA2000*   D2000*  Pmag~  \
0     3C273_Quasar_3C273_Typ_Qu  Quasar_3C273  187.2470   2.0648   12.9   
1   HD_108959_HD108959_Typ_F0IV      HD108959  187.7800   1.3394      8   
2   Rasalhague_HD_159561_Typ_A5     HD 159561  263.7019  12.5725   2.07   
3   Rasalgethi_HD_156014_Typ_M5     HD 156014  258.6300  14.4028    5.3   
4       Altair_HD_187642_Typ_A7     HD 187642  297.6645   8.8808   0.76   
5         Vega_HD_172167_Typ_A0     HD 172167  279.1951  38.7961  0.026   
6         Dubhe_HD_95689_Typ_K0      HD 95689  165.8666  61.7635   1.79   
7       Scheat_HD_217906_Typ_M2     HD 217906  345.9085  28.0952   2.42   
8        Mizar_HD_116656_Typ_A2     HD 116656  200.9276  54.9378   2.04   
9        Alcor_HD_116657_Typ_MK     HD 116657  201.2525  55.0004   3.88   
10       R_Lyr_HD_175865_Typ_M5     HD 175865  283.7908  43.9585    3.9   
11   Alpheratz_HD_358_Typ_B8_A7        HD 358    2.0615  29.1029   2.06   
12     Albireo_HD_183912_

# CELESTIAL COORDINATES FOR NEW STAR Zet1 Lyr

In [102]:
# CELESTIAL COORDINATES FOR NEW STAR

from IPython.display import Markdown as md
# Instead of setting the cell to Markdown, create Markdown from withnin a code cell!
# We can just use python variable replacement syntax to make the text dynamic
target_name = "Zet1 Lyr"
md(f"### Obtain spectral details for new star {target_name}")


### Obtain spectral details for new star Zet1 Lyr

In [103]:
new_coord, orig_coord = obtain_info_for_star(target_name)

<SkyCoord (ICRS): (ra, dec) in deg
    (281.19315451, 37.60512165)>

sky_PA: 291.9 deg

offset_dist: 2.0 arcmin

dx: -1.8556725077978407 arcmin dy: 0.7459755651516162 arcmin

offset_frame = <SkyOffsetICRS Frame (rotation=0.0 deg, origin=<ICRS Coordinate: (ra, dec) in deg
    (281.19315451, 37.60512165)>)>

offset_coord = <SkyCoord (SkyOffsetICRS: rotation=0.0 deg, origin=<ICRS Coordinate: (ra, dec) in deg
    (281.19315451, 37.60512165)>): (lon, lat) in deg
    (-0.03092788, 0.01243293)>

new_coord_ = <SkyCoord (ICRS): (ra, dec) in deg
    (281.15410922, 37.61754815)>
Using SkyOffsetFrame for Star Zet1 Lyr 
User inputs: offset_arcmin = 2, camera_rotation_deg = -21.9

Original RA(Hr)/Dec: 18.7462, 37.6051
Original RA(Deg)/Dec: 281.1932, 37.6051
New  RA(Hr)/Dec:  18.7436, 37.6175
New  RA(Deg)/Dec:  281.1541, 37.6175
Delta RA/DEC(min): 0.0390,           -0.0124


### Update Properties for Star based on wiki query results above 

In [104]:
star_magnitude = 4.37;  target_alt_name = "HD 173648" # FIX THIS LINE AND BELOW BASED ON SEARCH
star_type = 'kA5'
exposure = compute_exposure_time(star_magnitude)
print(f"Required exposure for mag {star_magnitude}: {exposure:.2f} s")

Required exposure for mag 4.37: 28.44 s


In [105]:
df.loc[ridx]=[f'{target_name}_{target_alt_name}_Typ_{star_type}',f'{target_alt_name}',f'{new_coord.ra.deg:.4f}',f'{new_coord.dec.deg:.4f}',
           f'{star_magnitude}',f'{exposure:.2f}','NA','NA','1','0','',f'{orig_coord.ra.hour:.4f}',f'{orig_coord.ra.deg:.4f}',f'{orig_coord.dec.deg:.4f}']
ridx += 1
df['Name1*'] = df['Name1*'].str.replace(' ', '_'); print(df[['Name1*','Name2*',"RA2000*","D2000*","Pmag~","Exp~"]])

                         Name1*        Name2*   RA2000*   D2000*  Pmag~  \
0     3C273_Quasar_3C273_Typ_Qu  Quasar_3C273  187.2470   2.0648   12.9   
1   HD_108959_HD108959_Typ_F0IV      HD108959  187.7800   1.3394      8   
2   Rasalhague_HD_159561_Typ_A5     HD 159561  263.7019  12.5725   2.07   
3   Rasalgethi_HD_156014_Typ_M5     HD 156014  258.6300  14.4028    5.3   
4       Altair_HD_187642_Typ_A7     HD 187642  297.6645   8.8808   0.76   
5         Vega_HD_172167_Typ_A0     HD 172167  279.1951  38.7961  0.026   
6         Dubhe_HD_95689_Typ_K0      HD 95689  165.8666  61.7635   1.79   
7       Scheat_HD_217906_Typ_M2     HD 217906  345.9085  28.0952   2.42   
8        Mizar_HD_116656_Typ_A2     HD 116656  200.9276  54.9378   2.04   
9        Alcor_HD_116657_Typ_MK     HD 116657  201.2525  55.0004   3.88   
10       R_Lyr_HD_175865_Typ_M5     HD 175865  283.7908  43.9585    3.9   
11   Alpheratz_HD_358_Typ_B8_A7        HD 358    2.0615  29.1029   2.06   
12     Albireo_HD_183912_

# CELESTIAL COORDINATES FOR NEW STAR Zet2 Lyr

In [106]:
# CELESTIAL COORDINATES FOR NEW STAR

from IPython.display import Markdown as md
# Instead of setting the cell to Markdown, create Markdown from withnin a code cell!
# We can just use python variable replacement syntax to make the text dynamic
target_name = "Zet2 Lyr"
md(f"### Obtain spectral details for new star {target_name}")


### Obtain spectral details for new star Zet2 Lyr

In [107]:
new_coord, orig_coord = obtain_info_for_star(target_name)

<SkyCoord (ICRS): (ra, dec) in deg
    (281.20082994, 37.5945996)>

sky_PA: 291.9 deg

offset_dist: 2.0 arcmin

dx: -1.8556725077978407 arcmin dy: 0.7459755651516162 arcmin

offset_frame = <SkyOffsetICRS Frame (rotation=0.0 deg, origin=<ICRS Coordinate: (ra, dec) in deg
    (281.20082994, 37.5945996)>)>

offset_coord = <SkyCoord (SkyOffsetICRS: rotation=0.0 deg, origin=<ICRS Coordinate: (ra, dec) in deg
    (281.20082994, 37.5945996)>): (lon, lat) in deg
    (-0.03092788, 0.01243293)>

new_coord_ = <SkyCoord (ICRS): (ra, dec) in deg
    (281.16179019, 37.60702609)>
Using SkyOffsetFrame for Star Zet2 Lyr 
User inputs: offset_arcmin = 2, camera_rotation_deg = -21.9

Original RA(Hr)/Dec: 18.7467, 37.5946
Original RA(Deg)/Dec: 281.2008, 37.5946
New  RA(Hr)/Dec:  18.7441, 37.6070
New  RA(Deg)/Dec:  281.1618, 37.6070
Delta RA/DEC(min): 0.0390,           -0.0124


### Update Properties for Star based on wiki query results above 

In [108]:
star_magnitude = 5.74;  target_alt_name = "HD 173649" # FIX THIS LINE AND BELOW BASED ON SEARCH
star_type = 'F0'
exposure = compute_exposure_time(star_magnitude)
print(f"Required exposure for mag {star_magnitude}: {exposure:.2f} s")

Required exposure for mag 5.74: 95.25 s


In [109]:
df.loc[ridx]=[f'{target_name}_{target_alt_name}_Typ_{star_type}',f'{target_alt_name}',f'{new_coord.ra.deg:.4f}',f'{new_coord.dec.deg:.4f}',
           f'{star_magnitude}',f'{exposure:.2f}','NA','NA','1','0','',f'{orig_coord.ra.hour:.4f}',f'{orig_coord.ra.deg:.4f}',f'{orig_coord.dec.deg:.4f}']
ridx += 1
df['Name1*'] = df['Name1*'].str.replace(' ', '_'); print(df[['Name1*','Name2*',"RA2000*","D2000*","Pmag~","Exp~"]])

                         Name1*        Name2*   RA2000*   D2000*  Pmag~  \
0     3C273_Quasar_3C273_Typ_Qu  Quasar_3C273  187.2470   2.0648   12.9   
1   HD_108959_HD108959_Typ_F0IV      HD108959  187.7800   1.3394      8   
2   Rasalhague_HD_159561_Typ_A5     HD 159561  263.7019  12.5725   2.07   
3   Rasalgethi_HD_156014_Typ_M5     HD 156014  258.6300  14.4028    5.3   
4       Altair_HD_187642_Typ_A7     HD 187642  297.6645   8.8808   0.76   
5         Vega_HD_172167_Typ_A0     HD 172167  279.1951  38.7961  0.026   
6         Dubhe_HD_95689_Typ_K0      HD 95689  165.8666  61.7635   1.79   
7       Scheat_HD_217906_Typ_M2     HD 217906  345.9085  28.0952   2.42   
8        Mizar_HD_116656_Typ_A2     HD 116656  200.9276  54.9378   2.04   
9        Alcor_HD_116657_Typ_MK     HD 116657  201.2525  55.0004   3.88   
10       R_Lyr_HD_175865_Typ_M5     HD 175865  283.7908  43.9585    3.9   
11   Alpheratz_HD_358_Typ_B8_A7        HD 358    2.0615  29.1029   2.06   
12     Albireo_HD_183912_

# CELESTIAL COORDINATES FOR NEW STAR Kochab

In [110]:
# CELESTIAL COORDINATES FOR NEW STAR

from IPython.display import Markdown as md
# Instead of setting the cell to Markdown, create Markdown from withnin a code cell!
# We can just use python variable replacement syntax to make the text dynamic
target_name = "Kochab"
md(f"### Obtain spectral details for new star {target_name}")


### Obtain spectral details for new star Kochab

In [111]:
new_coord, orig_coord = obtain_info_for_star(target_name)

<SkyCoord (ICRS): (ra, dec) in deg
    (222.6763575, 74.15550394)>

sky_PA: 291.9 deg

offset_dist: 2.0 arcmin

dx: -1.8556725077978407 arcmin dy: 0.7459755651516162 arcmin

offset_frame = <SkyOffsetICRS Frame (rotation=0.0 deg, origin=<ICRS Coordinate: (ra, dec) in deg
    (222.6763575, 74.15550394)>)>

offset_coord = <SkyCoord (SkyOffsetICRS: rotation=0.0 deg, origin=<ICRS Coordinate: (ra, dec) in deg
    (222.6763575, 74.15550394)>): (lon, lat) in deg
    (-0.03092788, 0.01243293)>

new_coord_ = <SkyCoord (ICRS): (ra, dec) in deg
    (222.56299342, 74.16790743)>
Using SkyOffsetFrame for Star Kochab 
User inputs: offset_arcmin = 2, camera_rotation_deg = -21.9

Original RA(Hr)/Dec: 14.8451, 74.1555
Original RA(Deg)/Dec: 222.6764, 74.1555
New  RA(Hr)/Dec:  14.8375, 74.1679
New  RA(Deg)/Dec:  222.5630, 74.1679
Delta RA/DEC(min): 0.1134,           -0.0124


### Update Properties for Star based on wiki query results above 

In [112]:
star_magnitude = 2.08;  target_alt_name = "HD 131873" # FIX THIS LINE AND BELOW BASED ON SEARCH
star_type = 'K4'
exposure = compute_exposure_time(star_magnitude)
print(f"Required exposure for mag {star_magnitude}: {exposure:.2f} s")

Required exposure for mag 2.08: 3.77 s


In [113]:
df.loc[ridx]=[f'{target_name}_{target_alt_name}_Typ_{star_type}',f'{target_alt_name}',f'{new_coord.ra.deg:.4f}',f'{new_coord.dec.deg:.4f}',
           f'{star_magnitude}',f'{exposure:.2f}','NA','NA','1','0','',f'{orig_coord.ra.hour:.4f}',f'{orig_coord.ra.deg:.4f}',f'{orig_coord.dec.deg:.4f}']
ridx += 1
df['Name1*'] = df['Name1*'].str.replace(' ', '_'); print(df[['Name1*','Name2*',"RA2000*","D2000*","Pmag~","Exp~"]])

                         Name1*        Name2*   RA2000*   D2000*  Pmag~  \
0     3C273_Quasar_3C273_Typ_Qu  Quasar_3C273  187.2470   2.0648   12.9   
1   HD_108959_HD108959_Typ_F0IV      HD108959  187.7800   1.3394      8   
2   Rasalhague_HD_159561_Typ_A5     HD 159561  263.7019  12.5725   2.07   
3   Rasalgethi_HD_156014_Typ_M5     HD 156014  258.6300  14.4028    5.3   
4       Altair_HD_187642_Typ_A7     HD 187642  297.6645   8.8808   0.76   
5         Vega_HD_172167_Typ_A0     HD 172167  279.1951  38.7961  0.026   
6         Dubhe_HD_95689_Typ_K0      HD 95689  165.8666  61.7635   1.79   
7       Scheat_HD_217906_Typ_M2     HD 217906  345.9085  28.0952   2.42   
8        Mizar_HD_116656_Typ_A2     HD 116656  200.9276  54.9378   2.04   
9        Alcor_HD_116657_Typ_MK     HD 116657  201.2525  55.0004   3.88   
10       R_Lyr_HD_175865_Typ_M5     HD 175865  283.7908  43.9585    3.9   
11   Alpheratz_HD_358_Typ_B8_A7        HD 358    2.0615  29.1029   2.06   
12     Albireo_HD_183912_

# CELESTIAL COORDINATES FOR NEW STAR Dschubba

In [114]:
# CELESTIAL COORDINATES FOR NEW STAR

from IPython.display import Markdown as md
# Instead of setting the cell to Markdown, create Markdown from withnin a code cell!
# We can just use python variable replacement syntax to make the text dynamic
target_name = "Dschubba"
md(f"### Obtain spectral details for new star {target_name}")


### Obtain spectral details for new star Dschubba

In [115]:
new_coord, orig_coord = obtain_info_for_star(target_name)

<SkyCoord (ICRS): (ra, dec) in deg
    (240.08335535, -22.62170643)>

sky_PA: 291.9 deg

offset_dist: 2.0 arcmin

dx: -1.8556725077978407 arcmin dy: 0.7459755651516162 arcmin

offset_frame = <SkyOffsetICRS Frame (rotation=0.0 deg, origin=<ICRS Coordinate: (ra, dec) in deg
    (240.08335535, -22.62170643)>)>

offset_coord = <SkyCoord (SkyOffsetICRS: rotation=0.0 deg, origin=<ICRS Coordinate: (ra, dec) in deg
    (240.08335535, -22.62170643)>): (lon, lat) in deg
    (-0.03092788, 0.01243293)>

new_coord_ = <SkyCoord (ICRS): (ra, dec) in deg
    (240.04985273, -22.60927002)>
Using SkyOffsetFrame for Star Dschubba 
User inputs: offset_arcmin = 2, camera_rotation_deg = -21.9

Original RA(Hr)/Dec: 16.0056, -22.6217
Original RA(Deg)/Dec: 240.0834, -22.6217
New  RA(Hr)/Dec:  16.0033, -22.6093
New  RA(Deg)/Dec:  240.0499, -22.6093
Delta RA/DEC(min): 0.0335,           -0.0124


### Update Properties for Star based on wiki query results above 

In [116]:
star_magnitude = 1.59;  target_alt_name = "HD 143275" # FIX THIS LINE AND BELOW BASED ON SEARCH
star_type = 'B0'
exposure = compute_exposure_time(star_magnitude)
print(f"Required exposure for mag {star_magnitude}: {exposure:.2f} s")

Required exposure for mag 1.59: 2.45 s


In [117]:
df.loc[ridx]=[f'{target_name}_{target_alt_name}_Typ_{star_type}',f'{target_alt_name}',f'{new_coord.ra.deg:.4f}',f'{new_coord.dec.deg:.4f}',
           f'{star_magnitude}',f'{exposure:.2f}','NA','NA','1','0','',f'{orig_coord.ra.hour:.4f}',f'{orig_coord.ra.deg:.4f}',f'{orig_coord.dec.deg:.4f}']
ridx += 1
df['Name1*'] = df['Name1*'].str.replace(' ', '_'); print(df[['Name1*','Name2*',"RA2000*","D2000*","Pmag~","Exp~"]])

                         Name1*        Name2*   RA2000*    D2000*  Pmag~  \
0     3C273_Quasar_3C273_Typ_Qu  Quasar_3C273  187.2470    2.0648   12.9   
1   HD_108959_HD108959_Typ_F0IV      HD108959  187.7800    1.3394      8   
2   Rasalhague_HD_159561_Typ_A5     HD 159561  263.7019   12.5725   2.07   
3   Rasalgethi_HD_156014_Typ_M5     HD 156014  258.6300   14.4028    5.3   
4       Altair_HD_187642_Typ_A7     HD 187642  297.6645    8.8808   0.76   
5         Vega_HD_172167_Typ_A0     HD 172167  279.1951   38.7961  0.026   
6         Dubhe_HD_95689_Typ_K0      HD 95689  165.8666   61.7635   1.79   
7       Scheat_HD_217906_Typ_M2     HD 217906  345.9085   28.0952   2.42   
8        Mizar_HD_116656_Typ_A2     HD 116656  200.9276   54.9378   2.04   
9        Alcor_HD_116657_Typ_MK     HD 116657  201.2525   55.0004   3.88   
10       R_Lyr_HD_175865_Typ_M5     HD 175865  283.7908   43.9585    3.9   
11   Alpheratz_HD_358_Typ_B8_A7        HD 358    2.0615   29.1029   2.06   
12     Albir

# CELESTIAL COORDINATES FOR NEW STAR Enif

In [118]:
# CELESTIAL COORDINATES FOR NEW STAR

from IPython.display import Markdown as md
# Instead of setting the cell to Markdown, create Markdown from withnin a code cell!
# We can just use python variable replacement syntax to make the text dynamic
target_name = "Enif"
md(f"### Obtain spectral details for new star {target_name}")


### Obtain spectral details for new star Enif

In [119]:
new_coord, orig_coord = obtain_info_for_star(target_name)

<SkyCoord (ICRS): (ra, dec) in deg
    (326.04648391, 9.87500865)>

sky_PA: 291.9 deg

offset_dist: 2.0 arcmin

dx: -1.8556725077978407 arcmin dy: 0.7459755651516162 arcmin

offset_frame = <SkyOffsetICRS Frame (rotation=0.0 deg, origin=<ICRS Coordinate: (ra, dec) in deg
    (326.04648391, 9.87500865)>)>

offset_coord = <SkyCoord (SkyOffsetICRS: rotation=0.0 deg, origin=<ICRS Coordinate: (ra, dec) in deg
    (326.04648391, 9.87500865)>): (lon, lat) in deg
    (-0.03092788, 0.01243293)>

new_coord_ = <SkyCoord (ICRS): (ra, dec) in deg
    (326.01508974, 9.88744013)>
Using SkyOffsetFrame for Star Enif 
User inputs: offset_arcmin = 2, camera_rotation_deg = -21.9

Original RA(Hr)/Dec: 21.7364, 9.8750
Original RA(Deg)/Dec: 326.0465, 9.8750
New  RA(Hr)/Dec:  21.7343, 9.8874
New  RA(Deg)/Dec:  326.0151, 9.8874
Delta RA/DEC(min): 0.0314,           -0.0124


### Update Properties for Star based on wiki query results above 

In [120]:
star_magnitude = 2.37;  target_alt_name = "HD 206778" # FIX THIS LINE AND BELOW BASED ON SEARCH
star_type = 'K2'
exposure = compute_exposure_time(star_magnitude)
print(f"Required exposure for mag {star_magnitude}: {exposure:.2f} s")

Required exposure for mag 2.37: 4.87 s


In [121]:
df.loc[ridx]=[f'{target_name}_{target_alt_name}_Typ_{star_type}',f'{target_alt_name}',f'{new_coord.ra.deg:.4f}',f'{new_coord.dec.deg:.4f}',
           f'{star_magnitude}',f'{exposure:.2f}','NA','NA','1','0','',f'{orig_coord.ra.hour:.4f}',f'{orig_coord.ra.deg:.4f}',f'{orig_coord.dec.deg:.4f}']
ridx += 1
df['Name1*'] = df['Name1*'].str.replace(' ', '_'); print(df[['Name1*','Name2*',"RA2000*","D2000*","Pmag~","Exp~"]])

                         Name1*        Name2*   RA2000*    D2000*  Pmag~  \
0     3C273_Quasar_3C273_Typ_Qu  Quasar_3C273  187.2470    2.0648   12.9   
1   HD_108959_HD108959_Typ_F0IV      HD108959  187.7800    1.3394      8   
2   Rasalhague_HD_159561_Typ_A5     HD 159561  263.7019   12.5725   2.07   
3   Rasalgethi_HD_156014_Typ_M5     HD 156014  258.6300   14.4028    5.3   
4       Altair_HD_187642_Typ_A7     HD 187642  297.6645    8.8808   0.76   
5         Vega_HD_172167_Typ_A0     HD 172167  279.1951   38.7961  0.026   
6         Dubhe_HD_95689_Typ_K0      HD 95689  165.8666   61.7635   1.79   
7       Scheat_HD_217906_Typ_M2     HD 217906  345.9085   28.0952   2.42   
8        Mizar_HD_116656_Typ_A2     HD 116656  200.9276   54.9378   2.04   
9        Alcor_HD_116657_Typ_MK     HD 116657  201.2525   55.0004   3.88   
10       R_Lyr_HD_175865_Typ_M5     HD 175865  283.7908   43.9585    3.9   
11   Alpheratz_HD_358_Typ_B8_A7        HD 358    2.0615   29.1029   2.06   
12     Albir

# CELESTIAL COORDINATES FOR NEW STAR Alphecca

In [122]:
# CELESTIAL COORDINATES FOR NEW STAR

from IPython.display import Markdown as md
# Instead of setting the cell to Markdown, create Markdown from withnin a code cell!
# We can just use python variable replacement syntax to make the text dynamic
target_name = "Alphecca"
md(f"### Obtain spectral details for new star {target_name}")


### Obtain spectral details for new star Alphecca

In [123]:
new_coord, orig_coord = obtain_info_for_star(target_name)

<SkyCoord (ICRS): (ra, dec) in deg
    (233.67195203, 26.714685)>

sky_PA: 291.9 deg

offset_dist: 2.0 arcmin

dx: -1.8556725077978407 arcmin dy: 0.7459755651516162 arcmin

offset_frame = <SkyOffsetICRS Frame (rotation=0.0 deg, origin=<ICRS Coordinate: (ra, dec) in deg
    (233.67195203, 26.714685)>)>

offset_coord = <SkyCoord (SkyOffsetICRS: rotation=0.0 deg, origin=<ICRS Coordinate: (ra, dec) in deg
    (233.67195203, 26.714685)>): (lon, lat) in deg
    (-0.03092788, 0.01243293)>

new_coord_ = <SkyCoord (ICRS): (ra, dec) in deg
    (233.63732451, 26.72711372)>
Using SkyOffsetFrame for Star Alphecca 
User inputs: offset_arcmin = 2, camera_rotation_deg = -21.9

Original RA(Hr)/Dec: 15.5781, 26.7147
Original RA(Deg)/Dec: 233.6720, 26.7147
New  RA(Hr)/Dec:  15.5758, 26.7271
New  RA(Deg)/Dec:  233.6373, 26.7271
Delta RA/DEC(min): 0.0346,           -0.0124


### Update Properties for Star based on wiki query results above 

In [124]:
star_magnitude = 2.24;  target_alt_name = "HD 139006" # FIX THIS LINE AND BELOW BASED ON SEARCH
star_type = 'A0'
exposure = compute_exposure_time(star_magnitude)
print(f"Required exposure for mag {star_magnitude}: {exposure:.2f} s")

Required exposure for mag 2.24: 4.34 s


In [125]:
df.loc[ridx]=[f'{target_name}_{target_alt_name}_Typ_{star_type}',f'{target_alt_name}',f'{new_coord.ra.deg:.4f}',f'{new_coord.dec.deg:.4f}',
           f'{star_magnitude}',f'{exposure:.2f}','NA','NA','1','0','',f'{orig_coord.ra.hour:.4f}',f'{orig_coord.ra.deg:.4f}',f'{orig_coord.dec.deg:.4f}']
ridx += 1
df['Name1*'] = df['Name1*'].str.replace(' ', '_'); print(df[['Name1*','Name2*',"RA2000*","D2000*","Pmag~","Exp~"]])

                         Name1*        Name2*   RA2000*    D2000*  Pmag~  \
0     3C273_Quasar_3C273_Typ_Qu  Quasar_3C273  187.2470    2.0648   12.9   
1   HD_108959_HD108959_Typ_F0IV      HD108959  187.7800    1.3394      8   
2   Rasalhague_HD_159561_Typ_A5     HD 159561  263.7019   12.5725   2.07   
3   Rasalgethi_HD_156014_Typ_M5     HD 156014  258.6300   14.4028    5.3   
4       Altair_HD_187642_Typ_A7     HD 187642  297.6645    8.8808   0.76   
5         Vega_HD_172167_Typ_A0     HD 172167  279.1951   38.7961  0.026   
6         Dubhe_HD_95689_Typ_K0      HD 95689  165.8666   61.7635   1.79   
7       Scheat_HD_217906_Typ_M2     HD 217906  345.9085   28.0952   2.42   
8        Mizar_HD_116656_Typ_A2     HD 116656  200.9276   54.9378   2.04   
9        Alcor_HD_116657_Typ_MK     HD 116657  201.2525   55.0004   3.88   
10       R_Lyr_HD_175865_Typ_M5     HD 175865  283.7908   43.9585    3.9   
11   Alpheratz_HD_358_Typ_B8_A7        HD 358    2.0615   29.1029   2.06   
12     Albir

# CELESTIAL COORDINATES FOR NEW STAR Eltanin

In [126]:
# CELESTIAL COORDINATES FOR NEW STAR

from IPython.display import Markdown as md
# Instead of setting the cell to Markdown, create Markdown from withnin a code cell!
# We can just use python variable replacement syntax to make the text dynamic
target_name = "Eltanin"
md(f"### Obtain spectral details for new star {target_name}")


### Obtain spectral details for new star Eltanin

In [127]:
new_coord, orig_coord = obtain_info_for_star(target_name)

<SkyCoord (ICRS): (ra, dec) in deg
    (269.15154118, 51.48889562)>

sky_PA: 291.9 deg

offset_dist: 2.0 arcmin

dx: -1.8556725077978407 arcmin dy: 0.7459755651516162 arcmin

offset_frame = <SkyOffsetICRS Frame (rotation=0.0 deg, origin=<ICRS Coordinate: (ra, dec) in deg
    (269.15154118, 51.48889562)>)>

offset_coord = <SkyCoord (SkyOffsetICRS: rotation=0.0 deg, origin=<ICRS Coordinate: (ra, dec) in deg
    (269.15154118, 51.48889562)>): (lon, lat) in deg
    (-0.03092788, 0.01243293)>

new_coord_ = <SkyCoord (ICRS): (ra, dec) in deg
    (269.10185757, 51.50131805)>
Using SkyOffsetFrame for Star Eltanin 
User inputs: offset_arcmin = 2, camera_rotation_deg = -21.9

Original RA(Hr)/Dec: 17.9434, 51.4889
Original RA(Deg)/Dec: 269.1515, 51.4889
New  RA(Hr)/Dec:  17.9401, 51.5013
New  RA(Deg)/Dec:  269.1019, 51.5013
Delta RA/DEC(min): 0.0497,           -0.0124


### Update Properties for Star based on wiki query results above 

In [128]:
star_magnitude = 2.23;  target_alt_name = "HD 164058" # FIX THIS LINE AND BELOW BASED ON SEARCH
star_type = 'K5'
exposure = compute_exposure_time(star_magnitude)
print(f"Required exposure for mag {star_magnitude}: {exposure:.2f} s")

Required exposure for mag 2.23: 4.30 s


In [129]:
df.loc[ridx]=[f'{target_name}_{target_alt_name}_Typ_{star_type}',f'{target_alt_name}',f'{new_coord.ra.deg:.4f}',f'{new_coord.dec.deg:.4f}',
           f'{star_magnitude}',f'{exposure:.2f}','NA','NA','1','0','',f'{orig_coord.ra.hour:.4f}',f'{orig_coord.ra.deg:.4f}',f'{orig_coord.dec.deg:.4f}']
ridx += 1
df['Name1*'] = df['Name1*'].str.replace(' ', '_'); print(df[['Name1*','Name2*',"RA2000*","D2000*","Pmag~","Exp~"]])

                         Name1*        Name2*   RA2000*    D2000*  Pmag~  \
0     3C273_Quasar_3C273_Typ_Qu  Quasar_3C273  187.2470    2.0648   12.9   
1   HD_108959_HD108959_Typ_F0IV      HD108959  187.7800    1.3394      8   
2   Rasalhague_HD_159561_Typ_A5     HD 159561  263.7019   12.5725   2.07   
3   Rasalgethi_HD_156014_Typ_M5     HD 156014  258.6300   14.4028    5.3   
4       Altair_HD_187642_Typ_A7     HD 187642  297.6645    8.8808   0.76   
5         Vega_HD_172167_Typ_A0     HD 172167  279.1951   38.7961  0.026   
6         Dubhe_HD_95689_Typ_K0      HD 95689  165.8666   61.7635   1.79   
7       Scheat_HD_217906_Typ_M2     HD 217906  345.9085   28.0952   2.42   
8        Mizar_HD_116656_Typ_A2     HD 116656  200.9276   54.9378   2.04   
9        Alcor_HD_116657_Typ_MK     HD 116657  201.2525   55.0004   3.88   
10       R_Lyr_HD_175865_Typ_M5     HD 175865  283.7908   43.9585    3.9   
11   Alpheratz_HD_358_Typ_B8_A7        HD 358    2.0615   29.1029   2.06   
12     Albir

# CELESTIAL COORDINATES FOR NEW STAR Thuban

In [130]:
# CELESTIAL COORDINATES FOR NEW STAR

from IPython.display import Markdown as md
# Instead of setting the cell to Markdown, create Markdown from withnin a code cell!
# We can just use python variable replacement syntax to make the text dynamic
target_name = "Thuban"
md(f"### Obtain spectral details for new star {target_name}")


### Obtain spectral details for new star Thuban

In [131]:
new_coord, orig_coord = obtain_info_for_star(target_name)

<SkyCoord (ICRS): (ra, dec) in deg
    (211.09732332, 64.37586962)>

sky_PA: 291.9 deg

offset_dist: 2.0 arcmin

dx: -1.8556725077978407 arcmin dy: 0.7459755651516162 arcmin

offset_frame = <SkyOffsetICRS Frame (rotation=0.0 deg, origin=<ICRS Coordinate: (ra, dec) in deg
    (211.09732332, 64.37586962)>)>

offset_coord = <SkyCoord (SkyOffsetICRS: rotation=0.0 deg, origin=<ICRS Coordinate: (ra, dec) in deg
    (211.09732332, 64.37586962)>): (lon, lat) in deg
    (-0.03092788, 0.01243293)>

new_coord_ = <SkyCoord (ICRS): (ra, dec) in deg
    (211.02577574, 64.38828513)>
Using SkyOffsetFrame for Star Thuban 
User inputs: offset_arcmin = 2, camera_rotation_deg = -21.9

Original RA(Hr)/Dec: 14.0732, 64.3759
Original RA(Deg)/Dec: 211.0973, 64.3759
New  RA(Hr)/Dec:  14.0684, 64.3883
New  RA(Deg)/Dec:  211.0258, 64.3883
Delta RA/DEC(min): 0.0715,           -0.0124


### Update Properties for Star based on wiki query results above 

In [132]:
star_magnitude = 3.67;  target_alt_name = "HD 123299" # FIX THIS LINE AND BELOW BASED ON SEARCH
star_type = 'A0'
exposure = compute_exposure_time(star_magnitude)
print(f"Required exposure for mag {star_magnitude}: {exposure:.2f} s")

Required exposure for mag 3.67: 15.33 s


In [133]:
df.loc[ridx]=[f'{target_name}_{target_alt_name}_Typ_{star_type}',f'{target_alt_name}',f'{new_coord.ra.deg:.4f}',f'{new_coord.dec.deg:.4f}',
           f'{star_magnitude}',f'{exposure:.2f}','NA','NA','1','0','',f'{orig_coord.ra.hour:.4f}',f'{orig_coord.ra.deg:.4f}',f'{orig_coord.dec.deg:.4f}']
ridx += 1
df['Name1*'] = df['Name1*'].str.replace(' ', '_'); print(df[['Name1*','Name2*',"RA2000*","D2000*","Pmag~","Exp~"]])

                         Name1*        Name2*   RA2000*    D2000*  Pmag~  \
0     3C273_Quasar_3C273_Typ_Qu  Quasar_3C273  187.2470    2.0648   12.9   
1   HD_108959_HD108959_Typ_F0IV      HD108959  187.7800    1.3394      8   
2   Rasalhague_HD_159561_Typ_A5     HD 159561  263.7019   12.5725   2.07   
3   Rasalgethi_HD_156014_Typ_M5     HD 156014  258.6300   14.4028    5.3   
4       Altair_HD_187642_Typ_A7     HD 187642  297.6645    8.8808   0.76   
5         Vega_HD_172167_Typ_A0     HD 172167  279.1951   38.7961  0.026   
6         Dubhe_HD_95689_Typ_K0      HD 95689  165.8666   61.7635   1.79   
7       Scheat_HD_217906_Typ_M2     HD 217906  345.9085   28.0952   2.42   
8        Mizar_HD_116656_Typ_A2     HD 116656  200.9276   54.9378   2.04   
9        Alcor_HD_116657_Typ_MK     HD 116657  201.2525   55.0004   3.88   
10       R_Lyr_HD_175865_Typ_M5     HD 175865  283.7908   43.9585    3.9   
11   Alpheratz_HD_358_Typ_B8_A7        HD 358    2.0615   29.1029   2.06   
12     Albir

# CELESTIAL COORDINATES FOR NEW STAR h Uma

In [134]:
# CELESTIAL COORDINATES FOR NEW STAR

from IPython.display import Markdown as md
# Instead of setting the cell to Markdown, create Markdown from withnin a code cell!
# We can just use python variable replacement syntax to make the text dynamic
target_name = "h Uma"
md(f"### Obtain spectral details for new star {target_name}")


### Obtain spectral details for new star h Uma

In [135]:
new_coord, orig_coord = obtain_info_for_star(target_name)

<SkyCoord (ICRS): (ra, dec) in deg
    (142.88211696, 63.06185995)>

sky_PA: 291.9 deg

offset_dist: 2.0 arcmin

dx: -1.8556725077978407 arcmin dy: 0.7459755651516162 arcmin

offset_frame = <SkyOffsetICRS Frame (rotation=0.0 deg, origin=<ICRS Coordinate: (ra, dec) in deg
    (142.88211696, 63.06185995)>)>

offset_coord = <SkyCoord (SkyOffsetICRS: rotation=0.0 deg, origin=<ICRS Coordinate: (ra, dec) in deg
    (142.88211696, 63.06185995)>): (lon, lat) in deg
    (-0.03092788, 0.01243293)>

new_coord_ = <SkyCoord (ICRS): (ra, dec) in deg
    (142.81381863, 63.07427644)>
Using SkyOffsetFrame for Star h Uma 
User inputs: offset_arcmin = 2, camera_rotation_deg = -21.9

Original RA(Hr)/Dec: 9.5255, 63.0619
Original RA(Deg)/Dec: 142.8821, 63.0619
New  RA(Hr)/Dec:  9.5209, 63.0743
New  RA(Deg)/Dec:  142.8138, 63.0743
Delta RA/DEC(min): 0.0683,           -0.0124


### Update Properties for Star based on wiki query results above 

In [136]:
star_magnitude = 3.65;  target_alt_name = "HD 81937" # FIX THIS LINE AND BELOW BASED ON SEARCH
star_type = 'F0'
exposure = compute_exposure_time(star_magnitude)
print(f"Required exposure for mag {star_magnitude}: {exposure:.2f} s")

Required exposure for mag 3.65: 15.07 s


In [137]:
df.loc[ridx]=[f'{target_name}_{target_alt_name}_Typ_{star_type}',f'{target_alt_name}',f'{new_coord.ra.deg:.4f}',f'{new_coord.dec.deg:.4f}',
           f'{star_magnitude}',f'{exposure:.2f}','NA','NA','1','0','',f'{orig_coord.ra.hour:.4f}',f'{orig_coord.ra.deg:.4f}',f'{orig_coord.dec.deg:.4f}']
ridx += 1
df['Name1*'] = df['Name1*'].str.replace(' ', '_'); print(df[['Name1*','Name2*',"RA2000*","D2000*","Pmag~","Exp~"]])

                         Name1*        Name2*   RA2000*    D2000*  Pmag~  \
0     3C273_Quasar_3C273_Typ_Qu  Quasar_3C273  187.2470    2.0648   12.9   
1   HD_108959_HD108959_Typ_F0IV      HD108959  187.7800    1.3394      8   
2   Rasalhague_HD_159561_Typ_A5     HD 159561  263.7019   12.5725   2.07   
3   Rasalgethi_HD_156014_Typ_M5     HD 156014  258.6300   14.4028    5.3   
4       Altair_HD_187642_Typ_A7     HD 187642  297.6645    8.8808   0.76   
5         Vega_HD_172167_Typ_A0     HD 172167  279.1951   38.7961  0.026   
6         Dubhe_HD_95689_Typ_K0      HD 95689  165.8666   61.7635   1.79   
7       Scheat_HD_217906_Typ_M2     HD 217906  345.9085   28.0952   2.42   
8        Mizar_HD_116656_Typ_A2     HD 116656  200.9276   54.9378   2.04   
9        Alcor_HD_116657_Typ_MK     HD 116657  201.2525   55.0004   3.88   
10       R_Lyr_HD_175865_Typ_M5     HD 175865  283.7908   43.9585    3.9   
11   Alpheratz_HD_358_Typ_B8_A7        HD 358    2.0615   29.1029   2.06   
12     Albir

# CELESTIAL COORDINATES FOR NEW STAR Theta Cep

In [138]:
# CELESTIAL COORDINATES FOR NEW STAR

from IPython.display import Markdown as md
# Instead of setting the cell to Markdown, create Markdown from withnin a code cell!
# We can just use python variable replacement syntax to make the text dynamic
target_name = "Theta Cep"
md(f"### Obtain spectral details for new star {target_name}")


### Obtain spectral details for new star Theta Cep

In [139]:
new_coord, orig_coord = obtain_info_for_star(target_name)

<SkyCoord (ICRS): (ra, dec) in deg
    (307.39543861, 62.99411033)>

sky_PA: 291.9 deg

offset_dist: 2.0 arcmin

dx: -1.8556725077978407 arcmin dy: 0.7459755651516162 arcmin

offset_frame = <SkyOffsetICRS Frame (rotation=0.0 deg, origin=<ICRS Coordinate: (ra, dec) in deg
    (307.39543861, 62.99411033)>)>

offset_coord = <SkyCoord (SkyOffsetICRS: rotation=0.0 deg, origin=<ICRS Coordinate: (ra, dec) in deg
    (307.39543861, 62.99411033)>): (lon, lat) in deg
    (-0.03092788, 0.01243293)>

new_coord_ = <SkyCoord (ICRS): (ra, dec) in deg
    (307.32729887, 63.00652688)>
Using SkyOffsetFrame for Star Theta Cep 
User inputs: offset_arcmin = 2, camera_rotation_deg = -21.9

Original RA(Hr)/Dec: 20.4930, 62.9941
Original RA(Deg)/Dec: 307.3954, 62.9941
New  RA(Hr)/Dec:  20.4885, 63.0065
New  RA(Deg)/Dec:  307.3273, 63.0065
Delta RA/DEC(min): 0.0681,           -0.0124


### Update Properties for Star based on wiki query results above 

In [140]:
star_magnitude = 4.22;  target_alt_name = "HD 195725" # FIX THIS LINE AND BELOW BASED ON SEARCH
star_type = 'A7'
exposure = compute_exposure_time(star_magnitude)
print(f"Required exposure for mag {star_magnitude}: {exposure:.2f} s")

Required exposure for mag 4.22: 24.91 s


In [141]:
df.loc[ridx]=[f'{target_name}_{target_alt_name}_Typ_{star_type}',f'{target_alt_name}',f'{new_coord.ra.deg:.4f}',f'{new_coord.dec.deg:.4f}',
           f'{star_magnitude}',f'{exposure:.2f}','NA','NA','1','0','',f'{orig_coord.ra.hour:.4f}',f'{orig_coord.ra.deg:.4f}',f'{orig_coord.dec.deg:.4f}']
ridx += 1
df['Name1*'] = df['Name1*'].str.replace(' ', '_'); print(df[['Name1*','Name2*',"RA2000*","D2000*","Pmag~","Exp~"]])

                         Name1*        Name2*   RA2000*    D2000*  Pmag~  \
0     3C273_Quasar_3C273_Typ_Qu  Quasar_3C273  187.2470    2.0648   12.9   
1   HD_108959_HD108959_Typ_F0IV      HD108959  187.7800    1.3394      8   
2   Rasalhague_HD_159561_Typ_A5     HD 159561  263.7019   12.5725   2.07   
3   Rasalgethi_HD_156014_Typ_M5     HD 156014  258.6300   14.4028    5.3   
4       Altair_HD_187642_Typ_A7     HD 187642  297.6645    8.8808   0.76   
5         Vega_HD_172167_Typ_A0     HD 172167  279.1951   38.7961  0.026   
6         Dubhe_HD_95689_Typ_K0      HD 95689  165.8666   61.7635   1.79   
7       Scheat_HD_217906_Typ_M2     HD 217906  345.9085   28.0952   2.42   
8        Mizar_HD_116656_Typ_A2     HD 116656  200.9276   54.9378   2.04   
9        Alcor_HD_116657_Typ_MK     HD 116657  201.2525   55.0004   3.88   
10       R_Lyr_HD_175865_Typ_M5     HD 175865  283.7908   43.9585    3.9   
11   Alpheratz_HD_358_Typ_B8_A7        HD 358    2.0615   29.1029   2.06   
12     Albir

# CELESTIAL COORDINATES FOR NEW STAR VZ Cam

In [142]:
# CELESTIAL COORDINATES FOR NEW STAR

from IPython.display import Markdown as md
# Instead of setting the cell to Markdown, create Markdown from withnin a code cell!
# We can just use python variable replacement syntax to make the text dynamic
target_name = "VZ Cam"
md(f"### Obtain spectral details for new star {target_name}")


### Obtain spectral details for new star VZ Cam

In [143]:
new_coord, orig_coord = obtain_info_for_star(target_name)

<SkyCoord (ICRS): (ra, dec) in deg
    (112.76861553, 82.41146709)>

sky_PA: 291.9 deg

offset_dist: 2.0 arcmin

dx: -1.8556725077978407 arcmin dy: 0.7459755651516162 arcmin

offset_frame = <SkyOffsetICRS Frame (rotation=0.0 deg, origin=<ICRS Coordinate: (ra, dec) in deg
    (112.76861553, 82.41146709)>)>

offset_coord = <SkyCoord (SkyOffsetICRS: rotation=0.0 deg, origin=<ICRS Coordinate: (ra, dec) in deg
    (112.76861553, 82.41146709)>): (lon, lat) in deg
    (-0.03092788, 0.01243293)>

new_coord_ = <SkyCoord (ICRS): (ra, dec) in deg
    (112.53403557, 82.42383726)>
Using SkyOffsetFrame for Star VZ Cam 
User inputs: offset_arcmin = 2, camera_rotation_deg = -21.9

Original RA(Hr)/Dec: 7.5179, 82.4115
Original RA(Deg)/Dec: 112.7686, 82.4115
New  RA(Hr)/Dec:  7.5023, 82.4238
New  RA(Deg)/Dec:  112.5340, 82.4238
Delta RA/DEC(min): 0.2346,           -0.0124


### Update Properties for Star based on wiki query results above 

In [144]:
star_magnitude = 4.92;  target_alt_name = "HD 55966" # FIX THIS LINE AND BELOW BASED ON SEARCH
star_type = 'M4'
exposure = compute_exposure_time(star_magnitude)
print(f"Required exposure for mag {star_magnitude}: {exposure:.2f} s")

Required exposure for mag 4.92: 46.20 s


In [145]:
df.loc[ridx]=[f'{target_name}_{target_alt_name}_Typ_{star_type}',f'{target_alt_name}',f'{new_coord.ra.deg:.4f}',f'{new_coord.dec.deg:.4f}',
           f'{star_magnitude}',f'{exposure:.2f}','NA','NA','1','0','',f'{orig_coord.ra.hour:.4f}',f'{orig_coord.ra.deg:.4f}',f'{orig_coord.dec.deg:.4f}']
ridx += 1
df['Name1*'] = df['Name1*'].str.replace(' ', '_'); print(df[['Name1*','Name2*',"RA2000*","D2000*","Pmag~","Exp~"]])

                         Name1*        Name2*   RA2000*    D2000*  Pmag~  \
0     3C273_Quasar_3C273_Typ_Qu  Quasar_3C273  187.2470    2.0648   12.9   
1   HD_108959_HD108959_Typ_F0IV      HD108959  187.7800    1.3394      8   
2   Rasalhague_HD_159561_Typ_A5     HD 159561  263.7019   12.5725   2.07   
3   Rasalgethi_HD_156014_Typ_M5     HD 156014  258.6300   14.4028    5.3   
4       Altair_HD_187642_Typ_A7     HD 187642  297.6645    8.8808   0.76   
5         Vega_HD_172167_Typ_A0     HD 172167  279.1951   38.7961  0.026   
6         Dubhe_HD_95689_Typ_K0      HD 95689  165.8666   61.7635   1.79   
7       Scheat_HD_217906_Typ_M2     HD 217906  345.9085   28.0952   2.42   
8        Mizar_HD_116656_Typ_A2     HD 116656  200.9276   54.9378   2.04   
9        Alcor_HD_116657_Typ_MK     HD 116657  201.2525   55.0004   3.88   
10       R_Lyr_HD_175865_Typ_M5     HD 175865  283.7908   43.9585    3.9   
11   Alpheratz_HD_358_Typ_B8_A7        HD 358    2.0615   29.1029   2.06   
12     Albir

# CELESTIAL COORDINATES FOR NEW STAR Erakis

In [146]:
from IPython.display import Markdown as md
# Instead of setting the cell to Markdown, create Markdown from withnin a code cell!
# We can just use python variable replacement syntax to make the text dynamic
target_name = "Erakis"
md(f"### Obtain spectral details for new star {target_name}")


### Obtain spectral details for new star Erakis

In [147]:
new_coord, orig_coord = obtain_info_for_star(target_name)

<SkyCoord (ICRS): (ra, dec) in deg
    (325.87691482, 58.78004609)>

sky_PA: 291.9 deg

offset_dist: 2.0 arcmin

dx: -1.8556725077978407 arcmin dy: 0.7459755651516162 arcmin

offset_frame = <SkyOffsetICRS Frame (rotation=0.0 deg, origin=<ICRS Coordinate: (ra, dec) in deg
    (325.87691482, 58.78004609)>)>

offset_coord = <SkyCoord (SkyOffsetICRS: rotation=0.0 deg, origin=<ICRS Coordinate: (ra, dec) in deg
    (325.87691482, 58.78004609)>): (lon, lat) in deg
    (-0.03092788, 0.01243293)>

new_coord_ = <SkyCoord (ICRS): (ra, dec) in deg
    (325.81722456, 58.79246524)>
Using SkyOffsetFrame for Star Erakis 
User inputs: offset_arcmin = 2, camera_rotation_deg = -21.9

Original RA(Hr)/Dec: 21.7251, 58.7800
Original RA(Deg)/Dec: 325.8769, 58.7800
New  RA(Hr)/Dec:  21.7211, 58.7925
New  RA(Deg)/Dec:  325.8172, 58.7925
Delta RA/DEC(min): 0.0597,           -0.0124


### Update Properties for Star based on wiki query results above 

In [148]:
star_magnitude = 4.08;  target_alt_name = "HD 206936" # FIX THIS LINE AND BELOW BASED ON SEARCH
star_type = 'M2'
exposure = compute_exposure_time(star_magnitude)
print(f"Required exposure for mag {star_magnitude}: {exposure:.2f} s")

Required exposure for mag 4.08: 22.02 s


In [149]:
df.loc[ridx]=[f'{target_name}_{target_alt_name}_Typ_{star_type}',f'{target_alt_name}',f'{new_coord.ra.deg:.4f}',f'{new_coord.dec.deg:.4f}',
           f'{star_magnitude}',f'{exposure:.2f}','NA','NA','1','0','',f'{orig_coord.ra.hour:.4f}',f'{orig_coord.ra.deg:.4f}',f'{orig_coord.dec.deg:.4f}']
ridx += 1
df['Name1*'] = df['Name1*'].str.replace(' ', '_'); print(df[['Name1*','Name2*',"RA2000*","D2000*","Pmag~","Exp~"]])

                         Name1*        Name2*   RA2000*    D2000*  Pmag~  \
0     3C273_Quasar_3C273_Typ_Qu  Quasar_3C273  187.2470    2.0648   12.9   
1   HD_108959_HD108959_Typ_F0IV      HD108959  187.7800    1.3394      8   
2   Rasalhague_HD_159561_Typ_A5     HD 159561  263.7019   12.5725   2.07   
3   Rasalgethi_HD_156014_Typ_M5     HD 156014  258.6300   14.4028    5.3   
4       Altair_HD_187642_Typ_A7     HD 187642  297.6645    8.8808   0.76   
5         Vega_HD_172167_Typ_A0     HD 172167  279.1951   38.7961  0.026   
6         Dubhe_HD_95689_Typ_K0      HD 95689  165.8666   61.7635   1.79   
7       Scheat_HD_217906_Typ_M2     HD 217906  345.9085   28.0952   2.42   
8        Mizar_HD_116656_Typ_A2     HD 116656  200.9276   54.9378   2.04   
9        Alcor_HD_116657_Typ_MK     HD 116657  201.2525   55.0004   3.88   
10       R_Lyr_HD_175865_Typ_M5     HD 175865  283.7908   43.9585    3.9   
11   Alpheratz_HD_358_Typ_B8_A7        HD 358    2.0615   29.1029   2.06   
12     Albir

# CELESTIAL COORDINATES FOR NEW STAR 42 Her

In [150]:
# CELESTIAL COORDINATES FOR NEW STAR

from IPython.display import Markdown as md
# Instead of setting the cell to Markdown, create Markdown from withnin a code cell!
# We can just use python variable replacement syntax to make the text dynamic
target_name = "42 Her"
md(f"### Obtain spectral details for new star {target_name}")


### Obtain spectral details for new star 42 Her

In [151]:
new_coord, orig_coord = obtain_info_for_star(target_name)

<SkyCoord (ICRS): (ra, dec) in deg
    (249.68685421, 48.92834233)>

sky_PA: 291.9 deg

offset_dist: 2.0 arcmin

dx: -1.8556725077978407 arcmin dy: 0.7459755651516162 arcmin

offset_frame = <SkyOffsetICRS Frame (rotation=0.0 deg, origin=<ICRS Coordinate: (ra, dec) in deg
    (249.68685421, 48.92834233)>)>

offset_coord = <SkyCoord (SkyOffsetICRS: rotation=0.0 deg, origin=<ICRS Coordinate: (ra, dec) in deg
    (249.68685421, 48.92834233)>): (lon, lat) in deg
    (-0.03092788, 0.01243293)>

new_coord_ = <SkyCoord (ICRS): (ra, dec) in deg
    (249.63976827, 48.94076567)>
Using SkyOffsetFrame for Star 42 Her 
User inputs: offset_arcmin = 2, camera_rotation_deg = -21.9

Original RA(Hr)/Dec: 16.6458, 48.9283
Original RA(Deg)/Dec: 249.6869, 48.9283
New  RA(Hr)/Dec:  16.6427, 48.9408
New  RA(Deg)/Dec:  249.6398, 48.9408
Delta RA/DEC(min): 0.0471,           -0.0124


### Update Properties for Star based on wiki query results above 

In [152]:
star_magnitude = 4.86;  target_alt_name = "HD 150450" # FIX THIS LINE AND BELOW BASED ON SEARCH
star_type = 'M2'
exposure = compute_exposure_time(star_magnitude)
print(f"Required exposure for mag {star_magnitude}: {exposure:.2f} s")

Required exposure for mag 4.86: 43.82 s


In [153]:
df.loc[ridx]=[f'{target_name}_{target_alt_name}_Typ_{star_type}',f'{target_alt_name}',f'{new_coord.ra.deg:.4f}',f'{new_coord.dec.deg:.4f}',
           f'{star_magnitude}',f'{exposure:.2f}','NA','NA','1','0','',f'{orig_coord.ra.hour:.4f}',f'{orig_coord.ra.deg:.4f}',f'{orig_coord.dec.deg:.4f}']
ridx += 1
df['Name1*'] = df['Name1*'].str.replace(' ', '_'); print(df[['Name1*','Name2*',"RA2000*","D2000*","Pmag~","Exp~"]])

                         Name1*        Name2*   RA2000*    D2000*  Pmag~  \
0     3C273_Quasar_3C273_Typ_Qu  Quasar_3C273  187.2470    2.0648   12.9   
1   HD_108959_HD108959_Typ_F0IV      HD108959  187.7800    1.3394      8   
2   Rasalhague_HD_159561_Typ_A5     HD 159561  263.7019   12.5725   2.07   
3   Rasalgethi_HD_156014_Typ_M5     HD 156014  258.6300   14.4028    5.3   
4       Altair_HD_187642_Typ_A7     HD 187642  297.6645    8.8808   0.76   
5         Vega_HD_172167_Typ_A0     HD 172167  279.1951   38.7961  0.026   
6         Dubhe_HD_95689_Typ_K0      HD 95689  165.8666   61.7635   1.79   
7       Scheat_HD_217906_Typ_M2     HD 217906  345.9085   28.0952   2.42   
8        Mizar_HD_116656_Typ_A2     HD 116656  200.9276   54.9378   2.04   
9        Alcor_HD_116657_Typ_MK     HD 116657  201.2525   55.0004   3.88   
10       R_Lyr_HD_175865_Typ_M5     HD 175865  283.7908   43.9585    3.9   
11   Alpheratz_HD_358_Typ_B8_A7        HD 358    2.0615   29.1029   2.06   
12     Albir

# CELESTIAL COORDINATES FOR NEW STAR V906 Her

In [154]:
# CELESTIAL COORDINATES FOR NEW STAR

from IPython.display import Markdown as md
# Instead of setting the cell to Markdown, create Markdown from withnin a code cell!
# We can just use python variable replacement syntax to make the text dynamic
target_name = "V906 Her"
md(f"### Obtain spectral details for new star {target_name}")


### Obtain spectral details for new star V906 Her

In [155]:
new_coord, orig_coord = obtain_info_for_star(target_name)

<SkyCoord (ICRS): (ra, dec) in deg
    (249.63558952, 48.8622953)>

sky_PA: 291.9 deg

offset_dist: 2.0 arcmin

dx: -1.8556725077978407 arcmin dy: 0.7459755651516162 arcmin

offset_frame = <SkyOffsetICRS Frame (rotation=0.0 deg, origin=<ICRS Coordinate: (ra, dec) in deg
    (249.63558952, 48.8622953)>)>

offset_coord = <SkyCoord (SkyOffsetICRS: rotation=0.0 deg, origin=<ICRS Coordinate: (ra, dec) in deg
    (249.63558952, 48.8622953)>): (lon, lat) in deg
    (-0.03092788, 0.01243293)>

new_coord_ = <SkyCoord (ICRS): (ra, dec) in deg
    (249.58856578, 48.87471867)>
Using SkyOffsetFrame for Star V906 Her 
User inputs: offset_arcmin = 2, camera_rotation_deg = -21.9

Original RA(Hr)/Dec: 16.6424, 48.8623
Original RA(Deg)/Dec: 249.6356, 48.8623
New  RA(Hr)/Dec:  16.6392, 48.8747
New  RA(Deg)/Dec:  249.5886, 48.8747
Delta RA/DEC(min): 0.0470,           -0.0124


### Update Properties for Star based on wiki query results above 

In [156]:
star_magnitude = 6.60;  target_alt_name = "HD 150409" # FIX THIS LINE AND BELOW BASED ON SEARCH
star_type = 'Ma'
exposure = compute_exposure_time(star_magnitude)
print(f"Required exposure for mag {star_magnitude}: {exposure:.2f} s")

Required exposure for mag 6.6: 203.42 s


In [157]:
df.loc[ridx]=[f'{target_name}_{target_alt_name}_Typ_{star_type}',f'{target_alt_name}',f'{new_coord.ra.deg:.4f}',f'{new_coord.dec.deg:.4f}',
           f'{star_magnitude}',f'{exposure:.2f}','NA','NA','1','0','',f'{orig_coord.ra.hour:.4f}',f'{orig_coord.ra.deg:.4f}',f'{orig_coord.dec.deg:.4f}']
ridx += 1
df['Name1*'] = df['Name1*'].str.replace(' ', '_'); print(df[['Name1*','Name2*',"RA2000*","D2000*","Pmag~","Exp~"]])

                         Name1*        Name2*   RA2000*    D2000*  Pmag~  \
0     3C273_Quasar_3C273_Typ_Qu  Quasar_3C273  187.2470    2.0648   12.9   
1   HD_108959_HD108959_Typ_F0IV      HD108959  187.7800    1.3394      8   
2   Rasalhague_HD_159561_Typ_A5     HD 159561  263.7019   12.5725   2.07   
3   Rasalgethi_HD_156014_Typ_M5     HD 156014  258.6300   14.4028    5.3   
4       Altair_HD_187642_Typ_A7     HD 187642  297.6645    8.8808   0.76   
5         Vega_HD_172167_Typ_A0     HD 172167  279.1951   38.7961  0.026   
6         Dubhe_HD_95689_Typ_K0      HD 95689  165.8666   61.7635   1.79   
7       Scheat_HD_217906_Typ_M2     HD 217906  345.9085   28.0952   2.42   
8        Mizar_HD_116656_Typ_A2     HD 116656  200.9276   54.9378   2.04   
9        Alcor_HD_116657_Typ_MK     HD 116657  201.2525   55.0004   3.88   
10       R_Lyr_HD_175865_Typ_M5     HD 175865  283.7908   43.9585    3.9   
11   Alpheratz_HD_358_Typ_B8_A7        HD 358    2.0615   29.1029   2.06   
12     Albir

# CELESTIAL COORDINATES FOR Planet Neptune

In [158]:
# CELESTIAL COORDINATES FOR NEW STAR

from IPython.display import Markdown as md
# Instead of setting the cell to Markdown, create Markdown from withnin a code cell!
# We can just use python variable replacement syntax to make the text dynamic
target_name = "Neptune"
md(f"### Obtain spectral details for Planet {target_name}")


### Obtain spectral details for Planet Neptune

In [159]:
#current 08/15/25 6:30 PM Pacific coordinates for Neptune
planet_ra = 0.079 #decimal deg
planet_dec = -1.344 #decimal deg
planet_coord = SkyCoord(ra=planet_ra * u.deg, dec=planet_dec * u.deg, frame='icrs')
new_coord, orig_coord = obtain_adjcoord_for_planet(target_name, planet_coord)

<SkyCoord (ICRS): (ra, dec) in deg
    (0.079, -1.344)>

sky_PA: 291.9 deg

offset_dist: 2.0 arcmin

dx: -1.8556725077978407 arcmin dy: 0.7459755651516162 arcmin

offset_frame = <SkyOffsetICRS Frame (rotation=0.0 deg, origin=<ICRS Coordinate: (ra, dec) in deg
    (0.079, -1.344)>)>

offset_coord = <SkyCoord (SkyOffsetICRS: rotation=0.0 deg, origin=<ICRS Coordinate: (ra, dec) in deg
    (0.079, -1.344)>): (lon, lat) in deg
    (-0.03092788, 0.01243293)>

new_coord_ = <SkyCoord (ICRS): (ra, dec) in deg
    (0.04806377, -1.33156688)>
Using SkyOffsetFrame for Star Neptune 
User inputs: offset_arcmin = 2, camera_rotation_deg = -21.9

Original RA(Hr)/Dec: 0.0053, -1.3440
Original RA(Deg)/Dec: 0.0790, -1.3440
New  RA(Hr)/Dec:  0.0032, -1.3316
New  RA(Deg)/Dec:  0.0481, -1.3316
Delta RA/DEC(min): 0.0309,           -0.0124


### Update Properties for Star based on wiki query results above 

In [160]:
star_magnitude = 7.8;  target_alt_name = "Neptune" # FIX THIS LINE AND BELOW BASED ON SEARCH
star_type = 'Pl'
exposure = compute_exposure_time(star_magnitude)
print(f"Required exposure for mag {star_magnitude}: {exposure:.2f} s")

Required exposure for mag 7.8: 586.41 s


In [161]:
df.loc[ridx]=[f'{target_name}_{target_alt_name}_Typ_{star_type}',f'{target_alt_name}',f'{new_coord.ra.deg:.4f}',f'{new_coord.dec.deg:.4f}',
           f'{star_magnitude}',f'{exposure:.2f}','NA','NA','1','0','',f'{orig_coord.ra.hour:.4f}',f'{orig_coord.ra.deg:.4f}',f'{orig_coord.dec.deg:.4f}']
ridx += 1
df['Name1*'] = df['Name1*'].str.replace(' ', '_'); print(df[['Name1*','Name2*',"RA2000*","D2000*","Pmag~","Exp~"]])

                         Name1*        Name2*   RA2000*    D2000*  Pmag~  \
0     3C273_Quasar_3C273_Typ_Qu  Quasar_3C273  187.2470    2.0648   12.9   
1   HD_108959_HD108959_Typ_F0IV      HD108959  187.7800    1.3394      8   
2   Rasalhague_HD_159561_Typ_A5     HD 159561  263.7019   12.5725   2.07   
3   Rasalgethi_HD_156014_Typ_M5     HD 156014  258.6300   14.4028    5.3   
4       Altair_HD_187642_Typ_A7     HD 187642  297.6645    8.8808   0.76   
5         Vega_HD_172167_Typ_A0     HD 172167  279.1951   38.7961  0.026   
6         Dubhe_HD_95689_Typ_K0      HD 95689  165.8666   61.7635   1.79   
7       Scheat_HD_217906_Typ_M2     HD 217906  345.9085   28.0952   2.42   
8        Mizar_HD_116656_Typ_A2     HD 116656  200.9276   54.9378   2.04   
9        Alcor_HD_116657_Typ_MK     HD 116657  201.2525   55.0004   3.88   
10       R_Lyr_HD_175865_Typ_M5     HD 175865  283.7908   43.9585    3.9   
11   Alpheratz_HD_358_Typ_B8_A7        HD 358    2.0615   29.1029   2.06   
12     Albir

# CELESTIAL COORDINATES FOR Planet Saturn

In [162]:
# CELESTIAL COORDINATES FOR NEW STAR

from IPython.display import Markdown as md
# Instead of setting the cell to Markdown, create Markdown from withnin a code cell!
# We can just use python variable replacement syntax to make the text dynamic
target_name = "Saturn"
md(f"### Obtain spectral details for Planet {target_name}")


### Obtain spectral details for Planet Saturn

In [163]:
#current 08/15/25 6:30 PM Pacific coordinates for Neptune
planet_ra = 1.6195 #decimal deg
planet_dec = -1.9441 #decimal deg
planet_coord = SkyCoord(ra=planet_ra * u.deg, dec=planet_dec * u.deg, frame='icrs')
new_coord, orig_coord = obtain_adjcoord_for_planet(target_name, planet_coord)

<SkyCoord (ICRS): (ra, dec) in deg
    (1.6195, -1.9441)>

sky_PA: 291.9 deg

offset_dist: 2.0 arcmin

dx: -1.8556725077978407 arcmin dy: 0.7459755651516162 arcmin

offset_frame = <SkyOffsetICRS Frame (rotation=0.0 deg, origin=<ICRS Coordinate: (ra, dec) in deg
    (1.6195, -1.9441)>)>

offset_coord = <SkyCoord (SkyOffsetICRS: rotation=0.0 deg, origin=<ICRS Coordinate: (ra, dec) in deg
    (1.6195, -1.9441)>): (lon, lat) in deg
    (-0.03092788, 0.01243293)>

new_coord_ = <SkyCoord (ICRS): (ra, dec) in deg
    (1.58855454, -1.93166679)>
Using SkyOffsetFrame for Star Saturn 
User inputs: offset_arcmin = 2, camera_rotation_deg = -21.9

Original RA(Hr)/Dec: 0.1080, -1.9441
Original RA(Deg)/Dec: 1.6195, -1.9441
New  RA(Hr)/Dec:  0.1059, -1.9317
New  RA(Deg)/Dec:  1.5886, -1.9317
Delta RA/DEC(min): 0.0309,           -0.0124


### Update Properties for Star based on wiki query results above 

In [164]:
star_magnitude = 0.99;  target_alt_name = "Saturn" # FIX THIS LINE AND BELOW BASED ON SEARCH
star_type = 'Pl'
exposure = compute_exposure_time(star_magnitude)
print(f"Required exposure for mag {star_magnitude}: {exposure:.2f} s")

Required exposure for mag 0.99: 1.44 s


In [165]:
df.loc[ridx]=[f'{target_name}_{target_alt_name}_Typ_{star_type}',f'{target_alt_name}',f'{new_coord.ra.deg:.4f}',f'{new_coord.dec.deg:.4f}',
           f'{star_magnitude}',f'{exposure:.2f}','NA','NA','1','0','',f'{orig_coord.ra.hour:.4f}',f'{orig_coord.ra.deg:.4f}',f'{orig_coord.dec.deg:.4f}']
ridx += 1
df['Name1*'] = df['Name1*'].str.replace(' ', '_'); print(df[['Name1*','Name2*',"RA2000*","D2000*","Pmag~","Exp~"]])

                         Name1*        Name2*   RA2000*    D2000*  Pmag~  \
0     3C273_Quasar_3C273_Typ_Qu  Quasar_3C273  187.2470    2.0648   12.9   
1   HD_108959_HD108959_Typ_F0IV      HD108959  187.7800    1.3394      8   
2   Rasalhague_HD_159561_Typ_A5     HD 159561  263.7019   12.5725   2.07   
3   Rasalgethi_HD_156014_Typ_M5     HD 156014  258.6300   14.4028    5.3   
4       Altair_HD_187642_Typ_A7     HD 187642  297.6645    8.8808   0.76   
5         Vega_HD_172167_Typ_A0     HD 172167  279.1951   38.7961  0.026   
6         Dubhe_HD_95689_Typ_K0      HD 95689  165.8666   61.7635   1.79   
7       Scheat_HD_217906_Typ_M2     HD 217906  345.9085   28.0952   2.42   
8        Mizar_HD_116656_Typ_A2     HD 116656  200.9276   54.9378   2.04   
9        Alcor_HD_116657_Typ_MK     HD 116657  201.2525   55.0004   3.88   
10       R_Lyr_HD_175865_Typ_M5     HD 175865  283.7908   43.9585    3.9   
11   Alpheratz_HD_358_Typ_B8_A7        HD 358    2.0615   29.1029   2.06   
12     Albir

# CELESTIAL COORDINATES FOR Planet Uranus

In [166]:
# CELESTIAL COORDINATES FOR Planet Uranus

from IPython.display import Markdown as md
# Instead of setting the cell to Markdown, create Markdown from withnin a code cell!
# We can just use python variable replacement syntax to make the text dynamic
target_name = "Uranus"
md(f"### Obtain spectral details for Planet {target_name}")


### Obtain spectral details for Planet Uranus

In [167]:
#current 08/15/25 6:30 PM Pacific coordinates for Uranus
planet_ra = 52.504 #decimal deg
planet_dec = 18.711 #decimal deg
planet_coord = SkyCoord(ra=planet_ra * u.deg, dec=planet_dec * u.deg, frame='icrs')
new_coord, orig_coord = obtain_adjcoord_for_planet(target_name, planet_coord)

<SkyCoord (ICRS): (ra, dec) in deg
    (52.504, 18.711)>

sky_PA: 291.9 deg

offset_dist: 2.0 arcmin

dx: -1.8556725077978407 arcmin dy: 0.7459755651516162 arcmin

offset_frame = <SkyOffsetICRS Frame (rotation=0.0 deg, origin=<ICRS Coordinate: (ra, dec) in deg
    (52.504, 18.711)>)>

offset_coord = <SkyCoord (SkyOffsetICRS: rotation=0.0 deg, origin=<ICRS Coordinate: (ra, dec) in deg
    (52.504, 18.711)>): (lon, lat) in deg
    (-0.03092788, 0.01243293)>

new_coord_ = <SkyCoord (ICRS): (ra, dec) in deg
    (52.47134394, 18.7234301)>
Using SkyOffsetFrame for Star Uranus 
User inputs: offset_arcmin = 2, camera_rotation_deg = -21.9

Original RA(Hr)/Dec: 3.5003, 18.7110
Original RA(Deg)/Dec: 52.5040, 18.7110
New  RA(Hr)/Dec:  3.4981, 18.7234
New  RA(Deg)/Dec:  52.4713, 18.7234
Delta RA/DEC(min): 0.0327,           -0.0124


### Update Properties for Star based on wiki query results above 

In [168]:
star_magnitude = 5.75;  target_alt_name = target_name # FIX THIS LINE AND BELOW BASED ON SEARCH
star_type = 'Pl'
exposure = compute_exposure_time(star_magnitude)
print(f"Required exposure for mag {star_magnitude}: {exposure:.2f} s")

Required exposure for mag 5.75: 96.09 s


In [169]:
df.loc[ridx]=[f'{target_name}_{target_alt_name}_Typ_{star_type}',f'{target_alt_name}',f'{new_coord.ra.deg:.4f}',f'{new_coord.dec.deg:.4f}',
           f'{star_magnitude}',f'{exposure:.2f}','NA','NA','1','0','',f'{orig_coord.ra.hour:.4f}',f'{orig_coord.ra.deg:.4f}',f'{orig_coord.dec.deg:.4f}']
ridx += 1
df['Name1*'] = df['Name1*'].str.replace(' ', '_'); print(df[['Name1*','Name2*',"RA2000*","D2000*","Pmag~","Exp~"]])

                         Name1*        Name2*   RA2000*    D2000*  Pmag~  \
0     3C273_Quasar_3C273_Typ_Qu  Quasar_3C273  187.2470    2.0648   12.9   
1   HD_108959_HD108959_Typ_F0IV      HD108959  187.7800    1.3394      8   
2   Rasalhague_HD_159561_Typ_A5     HD 159561  263.7019   12.5725   2.07   
3   Rasalgethi_HD_156014_Typ_M5     HD 156014  258.6300   14.4028    5.3   
4       Altair_HD_187642_Typ_A7     HD 187642  297.6645    8.8808   0.76   
5         Vega_HD_172167_Typ_A0     HD 172167  279.1951   38.7961  0.026   
6         Dubhe_HD_95689_Typ_K0      HD 95689  165.8666   61.7635   1.79   
7       Scheat_HD_217906_Typ_M2     HD 217906  345.9085   28.0952   2.42   
8        Mizar_HD_116656_Typ_A2     HD 116656  200.9276   54.9378   2.04   
9        Alcor_HD_116657_Typ_MK     HD 116657  201.2525   55.0004   3.88   
10       R_Lyr_HD_175865_Typ_M5     HD 175865  283.7908   43.9585    3.9   
11   Alpheratz_HD_358_Typ_B8_A7        HD 358    2.0615   29.1029   2.06   
12     Albir

# OUTPUT CONSOLIDATED CSV FILE FOR BARO

In [170]:
df.to_csv(f"{data_folder_name}/{output_csvfilename}")
print(df[["Name1*","Name2*","RA2000*","D2000*","Pmag~","Exp~","Note1","Note2","NExp~","GetRef","Temp","rahrdec","rahrdegdec","decdegdec"]])

                         Name1*        Name2*   RA2000*    D2000*  Pmag~  \
0     3C273_Quasar_3C273_Typ_Qu  Quasar_3C273  187.2470    2.0648   12.9   
1   HD_108959_HD108959_Typ_F0IV      HD108959  187.7800    1.3394      8   
2   Rasalhague_HD_159561_Typ_A5     HD 159561  263.7019   12.5725   2.07   
3   Rasalgethi_HD_156014_Typ_M5     HD 156014  258.6300   14.4028    5.3   
4       Altair_HD_187642_Typ_A7     HD 187642  297.6645    8.8808   0.76   
5         Vega_HD_172167_Typ_A0     HD 172167  279.1951   38.7961  0.026   
6         Dubhe_HD_95689_Typ_K0      HD 95689  165.8666   61.7635   1.79   
7       Scheat_HD_217906_Typ_M2     HD 217906  345.9085   28.0952   2.42   
8        Mizar_HD_116656_Typ_A2     HD 116656  200.9276   54.9378   2.04   
9        Alcor_HD_116657_Typ_MK     HD 116657  201.2525   55.0004   3.88   
10       R_Lyr_HD_175865_Typ_M5     HD 175865  283.7908   43.9585    3.9   
11   Alpheratz_HD_358_Typ_B8_A7        HD 358    2.0615   29.1029   2.06   
12     Albir

# STOP HERE ADHOC CELESTIAL COORDINATES FOR ADHOC TARGET

In [171]:
import ipywidgets as widgets
from IPython.display import display

text_input = widgets.Text(description='Enter Adhoc Target Name:')
output = widgets.Output()
current_text_value = ""

def on_text_change(change):
    global current_text_value
    current_text_value = change['new']
    with output:
        output.clear_output(wait=True)
        print(f"Current input value: {current_text_value}")

text_input.observe(on_text_change, names='value')
display(text_input, output)

# Create an float input widget for RA
ra_widget = widgets.FloatText(
    value=0.0000,
    description='Enter a RA in decimal degrees for adhoc target:',
    disabled=False
)

# Create an float input widget for DEC
dec_widget = widgets.FloatText(
    value=0.0000,
    description='Enter a DEC in decimal degrees for adhoc target:',
    disabled=False
)

display(ra_widget)
display(dec_widget)

# You can access the value in another cell or later in the same cell's execution
# with `num_widget.value`
# Example:
# my_number = num_widget.value
# print(f"The number entered is: {my_number}")


Text(value='', description='Enter Adhoc Target Name:')

Output()

FloatText(value=0.0, description='Enter a RA in decimal degrees for adhoc target:')

FloatText(value=0.0, description='Enter a DEC in decimal degrees for adhoc target:')

In [172]:
my_tgt = text_input.value
my_ra = ra_widget.value
my_dec = dec_widget.value
print(f"The values entered are: {my_tgt} & {my_ra} & {my_dec}")

The values entered are:  & 0.0 & 0.0


In [173]:
new_coord = obtain_info_for_adhoc_star(my_tgt, my_ra, my_dec)

<SkyCoord (ICRS): (ra, dec) in deg
    (0., 0.)>

sky_PA: 291.9 deg

offset_dist: 2.0 arcmin

dx: -1.8556725077978407 arcmin dy: 0.7459755651516162 arcmin

offset_frame = <SkyOffsetICRS Frame (rotation=0.0 deg, origin=<ICRS Coordinate: (ra, dec) in deg
    (0., 0.)>)>

offset_coord = <SkyCoord (SkyOffsetICRS: rotation=0.0 deg, origin=<ICRS Coordinate: (ra, dec) in deg
    (0., 0.)>): (lon, lat) in deg
    (-0.03092788, 0.01243293)>

new_coord_ = <SkyCoord (ICRS): (ra, dec) in deg
    (359.96907212, 0.01243293)>
Using SkyOffsetFrame for Star  
User inputs: offset_arcmin = 2, camera_rotation_deg = -21.9

Original RA(Hr)/Dec: 0.0000, 0.0000
Original RA(Deg)/Dec: 0.0000, 0.0000
New  RA(Hr)/Dec:  23.9979, 0.0124
New  RA(Deg)/Dec:  359.9691, 0.0124
Delta RA/DEC(min): -359.9691,           -0.0124


### Update Properties for Star based on wiki query results above 

In [174]:
star_magnitude = 12.9;  target_alt_name = "Quasar_3C273" # FIX THIS LINE AND BELOW BASED ON SEARCH
star_type = 'Qu'
exposure = compute_exposure_time(star_magnitude)
print(f"Required exposure for mag {star_magnitude}: {exposure:.2f} s")

Required exposure for mag 12.9: 52772.22 s


In [175]:
df.loc[ridx]=[f'{target_name}_{target_alt_name}_Typ_{star_type}',f'{target_alt_name}',f'{new_coord.ra.deg:.4f}',f'{new_coord.dec.deg:.4f}',
           f'{star_magnitude}',f'{exposure:.2f}','NA','NA','1','0','',f'{orig_coord.ra.hour:.4f}',f'{orig_coord.ra.deg:.4f}',f'{orig_coord.dec.deg:.4f}']
ridx += 1
df['Name1*'] = df['Name1*'].str.replace(' ', '_'); print(df[['Name1*','Name2*',"RA2000*","D2000*","Pmag~","Exp~"]])

                         Name1*        Name2*   RA2000*    D2000*  Pmag~  \
0     3C273_Quasar_3C273_Typ_Qu  Quasar_3C273  187.2470    2.0648   12.9   
1   HD_108959_HD108959_Typ_F0IV      HD108959  187.7800    1.3394      8   
2   Rasalhague_HD_159561_Typ_A5     HD 159561  263.7019   12.5725   2.07   
3   Rasalgethi_HD_156014_Typ_M5     HD 156014  258.6300   14.4028    5.3   
4       Altair_HD_187642_Typ_A7     HD 187642  297.6645    8.8808   0.76   
5         Vega_HD_172167_Typ_A0     HD 172167  279.1951   38.7961  0.026   
6         Dubhe_HD_95689_Typ_K0      HD 95689  165.8666   61.7635   1.79   
7       Scheat_HD_217906_Typ_M2     HD 217906  345.9085   28.0952   2.42   
8        Mizar_HD_116656_Typ_A2     HD 116656  200.9276   54.9378   2.04   
9        Alcor_HD_116657_Typ_MK     HD 116657  201.2525   55.0004   3.88   
10       R_Lyr_HD_175865_Typ_M5     HD 175865  283.7908   43.9585    3.9   
11   Alpheratz_HD_358_Typ_B8_A7        HD 358    2.0615   29.1029   2.06   
12     Albir

In [176]:
star_magnitude = 12.9;  target_alt_name = my_tgt # FIX THIS LINE AND BELOW BASED ON SEARCH
star_type = 'Qu'
exposure = compute_exposure_time(star_magnitude)
print(f"Required exposure for mag {star_magnitude}: {exposure:.2f} s")

Required exposure for mag 12.9: 52772.22 s


In [177]:
df.loc[ridx]=[f'{target_name}_{target_alt_name}_Typ_{star_type}',f'{target_alt_name}',f'{new_coord.ra.deg:.4f}',f'{new_coord.dec.deg:.4f}',
           f'{star_magnitude}',f'{exposure:.2f}','NA','NA','1','0','',f'{orig_coord.ra.hour:.4f}',f'{orig_coord.ra.deg:.4f}',f'{orig_coord.dec.deg:.4f}']
ridx += 1
df['Name1*'] = df['Name1*'].str.replace(' ', '_'); print(df[['Name1*','Name2*',"RA2000*","D2000*","Pmag~","Exp~"]])

                         Name1*        Name2*   RA2000*    D2000*  Pmag~  \
0     3C273_Quasar_3C273_Typ_Qu  Quasar_3C273  187.2470    2.0648   12.9   
1   HD_108959_HD108959_Typ_F0IV      HD108959  187.7800    1.3394      8   
2   Rasalhague_HD_159561_Typ_A5     HD 159561  263.7019   12.5725   2.07   
3   Rasalgethi_HD_156014_Typ_M5     HD 156014  258.6300   14.4028    5.3   
4       Altair_HD_187642_Typ_A7     HD 187642  297.6645    8.8808   0.76   
5         Vega_HD_172167_Typ_A0     HD 172167  279.1951   38.7961  0.026   
6         Dubhe_HD_95689_Typ_K0      HD 95689  165.8666   61.7635   1.79   
7       Scheat_HD_217906_Typ_M2     HD 217906  345.9085   28.0952   2.42   
8        Mizar_HD_116656_Typ_A2     HD 116656  200.9276   54.9378   2.04   
9        Alcor_HD_116657_Typ_MK     HD 116657  201.2525   55.0004   3.88   
10       R_Lyr_HD_175865_Typ_M5     HD 175865  283.7908   43.9585    3.9   
11   Alpheratz_HD_358_Typ_B8_A7        HD 358    2.0615   29.1029   2.06   
12     Albir